# NKP-Math + Sieve + Ledger + Real-claims test: clean run order (v3)
Run the cells **top to bottom**. Cells 1-8 only write files. Cells 9-17 run checks. Part G is the real-claims test.

**Rule:** after any AI edit, run cell 10 (the gate). `RESULT: ALL PASS` = safe to continue. Empty output or any FAIL = stop.

Needs only sympy, mpmath, numpy (preinstalled in Colab) and sqlite3 (part of Python).

## Part A - write the files (cells 1-8)

In [ ]:
%%writefile nkp_math.py
"""
NKP-Math: claim checking where claims are MATHEMATICAL OBJECTS and sources are COMPUTATIONS.

Text engines can only compare strings; this one asks "is it true?" and computes the answer.
Claim types (each returns a Verdict):
  identity(lhs, rhs)            exact algebra, else 120-digit testing at positive AND negative points (no domain stated = all reals)
  value(expr, "3.14159")        evaluate to 60 digits, compare to the claimed decimal at ITS precision
  exceeds(expr, x, lo, hi, c)   does |f(x)| exceed c somewhere?  interval arithmetic + branch-and-bound:
                                can PROVE "never" (sup bound) or FIND a witness
  dimensions("F = m*a", dims)   units check, with symbolic exponents ("consistent only if b = 0")
  coincidence(a, b, formula)    evidence in BITS = precision bits - formula bits - search bits
Statuses: PROVED, VERIFIED, SUPPORTED, CONSISTENT (accept) | REFUTED, ILL_FORMED, COINCIDENCE (reject)
          CONDITIONAL, UNDETERMINED, SIGNIFICANT (reported as-is)
A SourceLedger keeps the track record of whoever made each claim (NKP's source-reliability idea, but
with ground truth supplied by computation instead of consensus).

Needs sympy and mpmath (both preinstalled in Colab).   python nkp_math.py --test
"""
from __future__ import annotations

import math
import random
import re
import sys
import time
from collections import Counter
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import mpmath as mp
import sympy as sp

mp.mp.dps = 50
ACCEPT = {"PROVED", "VERIFIED", "SUPPORTED", "CONSISTENT", "SIGNIFICANT"}
REJECT = {"REFUTED", "ILL_FORMED", "COINCIDENCE"}

_FUNCS = ("sin", "cos", "tan", "exp", "log", "sqrt", "Abs", "atan", "sinh", "cosh", "tanh")
_GLOBALS = {k: getattr(sp, k) for k in ("Symbol", "Integer", "Float", "Rational", "Function", "Dummy") + _FUNCS}
_GLOBALS["__builtins__"] = {}
_LOCALS = {"phi": sp.GoldenRatio, "pi": sp.pi, "euler": sp.E}
_TRANSFORMS = sp.parsing.sympy_parser.standard_transformations + (sp.parsing.sympy_parser.convert_xor,)


def parse(s: str):
    """Parse text into a sympy expression.  Decimals become exact Rationals ("0.3333" is NOT a binary float)."""
    s = re.sub(r"(?<![\w.])(\d+\.\d+)(?![\w.])", r"Rational('\1')", s)
    return sp.parse_expr(s, local_dict=dict(_LOCALS), global_dict=dict(_GLOBALS), transformations=_TRANSFORMS)


@dataclass
class Verdict:
    status: str
    detail: str
    ms: float = 0.0

    @property
    def accepted(self) -> bool:
        return self.status in ACCEPT


class SourceLedger:
    """Track record of claim sources, scored by what computation found."""

    def __init__(self):
        self.c: Dict[str, Counter] = {}

    def record(self, source: Optional[str], status: str) -> None:
        if source is None:
            return
        k = self.c.setdefault(source, Counter())
        if status in ACCEPT:
            k["true"] += 1
        elif status in REJECT:
            k["false"] += 1
        else:
            k["open"] += 1

    def reliability(self, source: str) -> float:
        k = self.c.get(source, Counter())
        return (k["true"] + 1) / (k["true"] + k["false"] + 2)      # Beta(1,1) prior

    def table(self) -> List[Tuple[str, int, int, int, float]]:
        return sorted(((s, k["true"], k["false"], k["open"], self.reliability(s)) for s, k in self.c.items()),
                      key=lambda r: -r[4])


class NKPMath:
    def __init__(self, n_points: int = 12, seed: int = 0, node_limit: int = 4000):
        self.n_points, self.rng, self.node_limit = n_points, random.Random(seed), node_limit
        self.ledger = SourceLedger()

    def _done(self, v: Verdict, t0: float, source: Optional[str]) -> Verdict:
        v.ms = (time.perf_counter() - t0) * 1000
        self.ledger.record(source, v.status)
        return v

    # ------------------------------------------------------------------ identity
    @staticmethod
    def _rational_nonzero(expr, syms) -> bool:
        """True only if expr is PROVABLY a nonzero rational function with rational coefficients."""
        try:
            num, _ = sp.fraction(sp.cancel(expr))
            P = sp.Poly(num, *syms)
            return (not P.is_zero) and all(c.is_Rational for c in P.coeffs())
        except Exception:
            return False

    def identity(self, lhs: str, rhs: str, domain: Optional[Dict[str, Tuple[float, float]]] = None,
                 source: Optional[str] = None) -> Verdict:
        """Is lhs == rhs?  Without a declared domain the claim is read as 'for all real inputs': it is tested on
        positive AND negative values, and a claim true only for positives is reported CONDITIONAL."""
        t0 = time.perf_counter()
        L, R = parse(lhs), parse(rhs)
        d = L - R
        syms = sorted(d.free_symbols, key=str)
        if not syms:
            if sp.simplify(d) == 0:
                return self._done(Verdict("PROVED", "exact: the difference simplifies to 0"), t0, source)
            val = sp.N(d, 60)
            if abs(val) > sp.Float(10) ** -40:
                return self._done(Verdict("REFUTED", f"lhs - rhs = {sp.N(d, 12)}"), t0, source)
            return self._done(Verdict("SUPPORTED", "agrees to 40 digits (not proved symbolically)"), t0, source)
        # exact algebra first: catches differences of ANY size between rational expressions
        try:
            dc = sp.cancel(d)
        except Exception:
            dc = d
        if dc == 0:
            return self._done(Verdict("PROVED", "exact: the rational difference cancels to 0"), t0, source)
        if self._rational_nonzero(d, syms):
            return self._done(Verdict("REFUTED", f"exact: lhs - rhs = {sp.sstr(dc)[:70]}, provably nonzero"), t0, source)
        DIGITS, TOL = 120, mp.mpf(10) ** -90
        with mp.workdps(DIGITS):
            fL, fR = sp.lambdify(syms, L, modules="mpmath"), sp.lambdify(syms, R, modules="mpmath")

            def first_failure(dom, signed):
                n_pts = max(self.n_points, 2 ** len(syms)) if signed else self.n_points
                for i in range(n_pts):
                    pt = []
                    for k, s_ in enumerate(syms):
                        lo, hi = dom.get(str(s_), (0.1, 3.0))
                        v = mp.mpf(lo) + (mp.mpf(hi) - mp.mpf(lo)) * mp.mpf(self.rng.getrandbits(420)) / mp.mpf(2) ** 420
                        neg = ((i >> k) & 1) if i < 2 ** len(syms) else (self.rng.random() < 0.5)   # every sign pattern is tried
                        pt.append(-v if (signed and neg) else v)
                    a_, b_ = fL(*pt), fR(*pt)
                    if abs(a_ - b_) > TOL * max(1, abs(a_), abs(b_)):
                        where = ", ".join(f"{s_}={mp.nstr(v, 6)}" for s_, v in zip(syms, pt))
                        return f"{where}: lhs-rhs = {mp.nstr(a_ - b_, 6)}"
                return None

            fail = first_failure(domain or {}, signed=False)
            if fail:
                return self._done(Verdict("REFUTED", f"counterexample at {fail}"), t0, source)
            if domain is None:
                fail = first_failure({}, signed=True)
                if fail:
                    return self._done(Verdict("CONDITIONAL", f"holds for positive inputs only; fails at {fail}"), t0, source)
        if sp.simplify(d) == 0 or sp.simplify(sp.expand_trig(d)) == 0:
            return self._done(Verdict("PROVED", "symbolic: the difference simplifies to 0"), t0, source)
        return self._done(Verdict("SUPPORTED", f"agrees to 90 digits at {self.n_points} random points (positive and negative); not proved"), t0, source)

    # ------------------------------------------------------------------ value
    def value(self, expr: str, claimed: str, source: Optional[str] = None) -> Verdict:
        t0 = time.perf_counter()
        decimals = len(claimed.split(".")[1]) if "." in claimed else 0
        with mp.workdps(max(50, decimals + 30)):            # compute well beyond the digits being claimed
            v = mp.mpmathify(str(sp.N(parse(expr), decimals + 40)))
            tol = mp.mpf(10) ** (-decimals) / 2
            err = abs(v - mp.mpf(claimed))
            if err <= tol * (1 + mp.mpf(10) ** -15):
                return self._done(Verdict("VERIFIED", f"matches to the claimed {decimals} decimals (true value {mp.nstr(v, min(decimals + 3, 40))})"), t0, source)
            return self._done(Verdict("REFUTED", f"claimed {claimed[:50]}; true value {mp.nstr(v, min(decimals + 3, 40))} (error {mp.nstr(err, 3)})"), t0, source)

    # ------------------------------------------------------------------ bound (interval arithmetic)
    def exceeds(self, expr: str, var: str, lo, hi, c, source: Optional[str] = None) -> Verdict:
        """Claim: there is an x in [lo, hi] with |f(x)| > c.  Proves 'never' or finds a witness."""
        t0 = time.perf_counter()
        x = sp.Symbol(var)
        e = parse(expr).subs(sp.GoldenRatio, (1 + sp.sqrt(5)) / 2)
        iv = mp.iv
        ns = {n: getattr(iv, n) for n in ("sin", "cos", "tan", "exp", "log", "sqrt") if hasattr(iv, n)}
        ns.update({"Abs": abs, "pi": iv.pi, "e": iv.e, "mpf": iv.mpf})   # constants must be intervals too          # (mpmath intervals have no atan/sinh: such terms are unsupported)
        f_iv = sp.lambdify(x, e, modules=[ns, "mpmath"])
        f_pt = sp.lambdify(x, e, modules="mpmath")
        c = mp.mpf(str(c)) if isinstance(c, str) else mp.mpf(c)
        lo, hi = mp.mpf(lo), mp.mpf(hi)

        def upper(a, b):
            r = f_iv(iv.mpf([a, b]))
            return max(abs(r.a), abs(r.b)) if hasattr(r, "a") else abs(r)

        U = upper(lo, hi)
        if U <= c:
            return self._done(Verdict("REFUTED", f"proved by interval arithmetic: |f| <= {mp.nstr(U, 8)} <= {mp.nstr(c, 8)} on [{lo}, {hi}]"), t0, source)
        stack, nodes, width0 = [(lo, hi)], 0, hi - lo
        while stack and nodes < self.node_limit:
            a, b = stack.pop()
            nodes += 1
            if upper(a, b) <= c:
                continue
            m = (a + b) / 2
            fm = abs(f_pt(m))
            if fm > c:
                return self._done(Verdict("PROVED", f"witness x = {mp.nstr(m, 12)}: |f| = {mp.nstr(fm, 10)} > {mp.nstr(c, 8)}"), t0, source)
            if (b - a) < width0 * mp.mpf(10) ** -14:
                continue
            stack.extend([(a, m), (m, b)])
        if not stack:
            return self._done(Verdict("REFUTED", f"proved by branch-and-bound: |f| <= {mp.nstr(c, 8)} everywhere ({nodes} boxes)"), t0, source)
        return self._done(Verdict("UNDETERMINED", f"no witness and no proof after {nodes} boxes"), t0, source)

    # ------------------------------------------------------------------ dimensions
    def dimensions(self, equation: str, dims: Dict[str, object], source: Optional[str] = None) -> Verdict:
        t0 = time.perf_counter()
        D = {k: self._dimspec(v) for k, v in dims.items()}
        issues: List[str] = []
        constraints: List[sp.Expr] = []

        def sub(a, b):
            return {k: sp.simplify(a.get(k, 0) - b.get(k, 0)) for k in set(a) | set(b)}

        def same(a, b, ctx):
            for k, dv in sub(a, b).items():
                if dv == 0:
                    continue
                if dv.free_symbols:
                    constraints.append(dv)
                else:
                    issues.append(f"{ctx}: [{self._fmt(a)}] vs [{self._fmt(b)}]")
                    return

        def dim(e):
            if e.is_Number or e in (sp.pi, sp.E, sp.GoldenRatio):
                return {}
            if e.is_Symbol:
                return dict(D.get(str(e), {}))
            if e.is_Add:
                first = dim(e.args[0])
                for a in e.args[1:]:
                    same(first, dim(a), f"cannot add terms in {sp.sstr(e)}")
                return first
            if e.is_Mul:
                out: Dict[str, sp.Expr] = {}
                for a in e.args:
                    for k, v in dim(a).items():
                        out[k] = sp.simplify(out.get(k, 0) + v)
                return out
            if e.is_Pow:
                b, ex = e.args
                if dim(ex):
                    issues.append(f"exponent has units in {sp.sstr(e)}")
                bd = dim(b)
                return {k: sp.simplify(v * ex) for k, v in bd.items()} if bd else {}
            if e.func in (sp.sin, sp.cos, sp.tan, sp.exp, sp.log, sp.atan, sp.sinh, sp.cosh, sp.tanh):
                same({}, dim(e.args[0]), f"argument of {e.func.__name__} must be dimensionless, in {sp.sstr(e)}")
                return {}
            if e.func == sp.Abs:
                return dim(e.args[0])
            issues.append(f"unsupported term {sp.sstr(e)}")
            return {}

        lhs, rhs = equation.split("=")
        same(dim(parse(lhs)), dim(parse(rhs)), "both sides of the equation")
        if issues:
            return self._done(Verdict("ILL_FORMED", "; ".join(issues)), t0, source)
        if constraints:
            unknowns = sorted({s for c in constraints for s in c.free_symbols}, key=str)
            sol = sp.solve(constraints, unknowns, dict=True)
            if not sol:
                return self._done(Verdict("ILL_FORMED", f"no choice of {unknowns} makes the units agree"), t0, source)
            return self._done(Verdict("CONDITIONAL", f"units agree ONLY if {sol[0]}"), t0, source)
        return self._done(Verdict("CONSISTENT", "units agree"), t0, source)

    @staticmethod
    def _dimspec(v):
        if isinstance(v, dict):
            return {k: sp.nsimplify(x) for k, x in v.items()}
        out = {}
        for tok in str(v).split():
            m = re.fullmatch(r"([A-Za-z]+)(?:\^(-?[\d/]+))?", tok)
            out[m.group(1)] = sp.Rational(m.group(2)) if m.group(2) else sp.Integer(1)
        return out

    @staticmethod
    def _fmt(d):
        s = " ".join(f"{k}^{v}" for k, v in sorted(d.items()) if v != 0)
        return s or "dimensionless"

    # ------------------------------------------------------------------ coincidence meter
    @staticmethod
    def formula_bits(formula: str, n_leaf: int = 30, n_op: int = 6) -> float:
        e = parse(formula)
        leaves = [a for a in sp.preorder_traversal(e) if a.is_Atom]
        bits = 0.0
        for a in leaves:
            bits += math.log2(n_leaf)
            if a.is_Integer and abs(a) > 12:
                bits += math.log2(abs(int(a)) + 1)          # big integers cost more to specify
        return bits + sp.count_ops(e) * math.log2(n_op)

    def coincidence(self, a: float, b: float, formula: str = "", rel_unc: float = 0.0,
                    search_bits: float = 8.6, threshold: float = 10.0, source: Optional[str] = None) -> Verdict:
        """Is 'a is close to b' evidence of a real relation?  net bits = precision - formula - search."""
        t0 = time.perf_counter()
        rel = max(abs(a - b) / max(abs(a), abs(b)), rel_unc)
        prec = math.log2(1 / (2 * rel)) if rel > 0 else 64.0
        fb = self.formula_bits(formula) if formula else 0.0
        net = prec - fb - search_bits
        txt = f"match to {rel:.2e} = {prec:.1f} bits; formula costs {fb:.1f} bits; search freedom {search_bits:.1f} bits; net {net:+.1f} bits"
        return self._done(Verdict("SIGNIFICANT" if net >= threshold else "COINCIDENCE", txt), t0, source)


# ----------------------------------------------------------------------------- tests
def run_tests() -> None:
    m = NKPMath()
    assert m.identity("sin(x)**2 + cos(x)**2", "1").status == "PROVED"
    r = m.identity("sin(x)", "x - x**3/6")
    assert r.status == "REFUTED" and "counterexample" in r.detail
    assert m.identity("x", "x + 10**(-20)*x**2").status == "REFUTED"                    # a 1e-20 near-identity
    assert m.identity("3*0.3333", "1").status == "REFUTED"                               # 0.9999, exact decimal
    assert m.identity("3*Rational(1,3)", "1").status == "PROVED"
    assert m.identity("2*cos(pi/5)", "phi").status in ("PROVED", "SUPPORTED")
    assert m.identity("1 - 3*(10**4 - 1)/(3*10**4)", "10**(-4)").status == "PROVED"     # the 'flaw' F_n = 10^-n
    assert m.value("pi", "3.14159").status == "VERIFIED" and m.value("pi", "3.14160").status == "REFUTED"
    assert m.value("2*pi/phi**2", "2.3999632297").status == "VERIFIED"
    assert m.value("3**9", "19683").status == "VERIFIED"
    # the 'balanced gate' claim: potential exceeds 1/3 somewhere on [0, 10]
    g = m.exceeds("sin(x*pi)*phi**(x/10)/(pi*phi)", "x", 0, 10, "0.3333333333333333")
    assert g.status == "REFUTED" and "interval" in g.detail, g
    # a narrow peak that a coarse grid would miss
    p = m.exceeds("2*exp(-((x-3.14159)/0.001)**2)", "x", 0, 10, "1.9")
    assert p.status == "PROVED", p
    assert m.exceeds("2*exp(-((x-3.14159)/0.001)**2)", "x", 0, 10, "2.01").status == "REFUTED"
    base = {"m": "M", "a": "L T^-2", "v": "L T^-1", "d": "L", "t": "T", "F": "M L T^-2", "En": "M L^2 T^-2"}
    assert m.dimensions("F = m*a", base).status == "CONSISTENT"
    assert m.dimensions("En = m*v**2/2", base).status == "CONSISTENT"
    bad = m.dimensions("En = m*v", base)
    assert bad.status == "ILL_FORMED", bad
    assert m.dimensions("En = m*v**2 + m*v", base).status == "ILL_FORMED"
    assert m.dimensions("En = m*sin(d)", base).status == "ILL_FORMED"                    # sin of a length
    # Copilot's 'single equation': consistent with GR only if b*s = 0
    gr = {"Gmn": "L^-2", "G": "L^3 M^-1 T^-2", "c": "L T^-1", "Tmn": "M L^-1 T^-2", "X": "M L^-1 T^-2"}
    assert m.dimensions("Gmn = 8*pi*G/c**4*Tmn", gr).status == "CONSISTENT"
    cond = m.dimensions("Gmn = 8*pi*G/c**4*(I*sqrt(X))**(1+b)*Tmn/sqrt(X)", gr)
    assert cond.status == "CONDITIONAL" and "b: 0" in cond.detail, cond
    # coincidence meter: golden angle (deg) vs 1/alpha is a coincidence; a 1e-14 match to a simple formula is not
    ga = 360 * (1 - 1 / ((1 + 5 ** 0.5) / 2))
    assert m.coincidence(ga, 137.035999084, "360*(1-1/phi)").status == "COINCIDENCE"
    assert m.coincidence(3.141592653589793, 3.1415926535897936, "pi").status == "SIGNIFICANT"
    # ledger
    m2 = NKPMath()
    m2.identity("x", "x+1", source="chatA"); m2.identity("x", "x", source="chatA"); m2.identity("x", "x+1", source="chatB")
    assert m2.ledger.reliability("chatA") > m2.ledger.reliability("chatB")
    # -- fixes found by the red-team (nkp_math_redteam.py)
    assert m.identity("sqrt(x**2)", "x").status == "CONDITIONAL"                          # true for x>0 only
    assert m.identity("sqrt(x**2)", "x", domain={"x": (0.1, 3.0)}).status in ("PROVED", "SUPPORTED")
    assert m.identity("sqrt(x**4)", "x**2").accepted and m.identity("sqrt(x**2)", "Abs(x)").accepted
    assert m.identity("x", "x + 10**(-60)*x**2").status == "REFUTED"                      # exact algebra sees any size
    assert m.identity("y", "y + 10**(-70)*y**3").status == "REFUTED"
    assert m.identity("exp(x)", "exp(x) + 10**(-100)*sin(x)").status == "SUPPORTED"      # documented limit: < 1e-90 and non-rational
    from decimal import Decimal, getcontext
    getcontext().prec = 140
    with mp.workdps(120):
        pi55 = format(Decimal(mp.nstr(mp.pi, 100)).quantize(Decimal(1).scaleb(-55)), "f")
    assert m.value("pi", pi55).status == "VERIFIED"
    assert m.value("pi", pi55[:-1] + str((int(pi55[-1]) + 1) % 10)).status == "REFUTED"
    print("all 28 checks passed")


if __name__ == "__main__":
    run_tests() if "--test" in sys.argv else print(__doc__)


In [ ]:
%%writefile nkp_math_redteam.py
"""
Red-team set for NKP-Math: hand-written claims chosen to FOOL the engine (domain traps, precision limits,
hard-but-true identities, subtle falsehoods, near-integers, tricky bounds, unit traps, numerology).

Set A and set B contain DIFFERENT instances of the SAME categories.  Both were written before any fix:
fix the engine using A, then B tells you whether the fix generalizes (B is held out).
Expected verdict classes: accept | reject | conditional     (UNDETERMINED counts as "reject": no proof, no acceptance)

LABEL CORRECTION (disclosed): A1, A3 and B3 (domain traps with no domain stated) were first labelled "reject".
The fixed engine answers CONDITIONAL ("true for positive inputs only"), which is the precise verdict, so they
are relabelled "conditional".  Scores under the ORIGINAL labels: A 25/27, B 26/27, with zero false accepts.
Set C is fresh (written after the fixes) and carries the correct labels from the start.

    python nkp_math_redteam.py          run both sets
"""
from __future__ import annotations

import sys
from decimal import ROUND_HALF_UP, Decimal, getcontext

import mpmath as mp

from nkp_math import NKPMath

getcontext().prec = 140
POS = {"x": (0.1, 3.0), "y": (0.1, 3.0)}


def digits(const: str, k: int, delta: int = 0) -> str:
    """Independent computation of a constant to k decimals (mpmath at 160 digits), optionally last digit + delta."""
    with mp.workdps(160):
        v = {"pi": mp.pi, "e": mp.e, "sqrt2": mp.sqrt(2)}[const]
        d = Decimal(mp.nstr(v, 150))
    q = d.quantize(Decimal(1).scaleb(-k), rounding=ROUND_HALF_UP) + delta * Decimal(1).scaleb(-k)
    return format(q, "f")


def dm(kind, **kw):
    return kind, kw


RED_A = [
    ("A1  sqrt(x^2) = x, no domain stated", "identity", dict(lhs="sqrt(x**2)", rhs="x"), "conditional"),
    ("A2  sqrt(x^2) = x, declared x>0", "identity", dict(lhs="sqrt(x**2)", rhs="x", domain=POS), "accept"),
    ("A3  log(x^2) = 2 log x, no domain", "identity", dict(lhs="log(x**2)", rhs="2*log(x)"), "conditional"),
    ("A4  log(x^2) = 2 log x, declared x>0", "identity", dict(lhs="log(x**2)", rhs="2*log(x)", domain=POS), "accept"),
    ("A5  x = x + 1e-60 x^2", "identity", dict(lhs="x", rhs="x + 10**(-60)*x**2"), "reject"),
    ("A6  pi to 55 decimals (correct)", "value", dict(expr="pi", claimed=digits("pi", 55)), "accept"),
    ("A7  pi to 55 decimals (last digit +1)", "value", dict(expr="pi", claimed=digits("pi", 55, +1)), "reject"),
    ("A8  sin3x = 3sinx - 4sin^3x", "identity", dict(lhs="sin(3*x)", rhs="3*sin(x)-4*sin(x)**3"), "accept"),
    ("A9  cos^4 - sin^4 = cos2x", "identity", dict(lhs="cos(x)**4-sin(x)**4", rhs="cos(2*x)"), "accept"),
    ("A10 cos5x = 16c^5-20c^3+5c", "identity", dict(lhs="cos(5*x)", rhs="16*cos(x)**5-20*cos(x)**3+5*cos(x)"), "accept"),
    ("A11 e^(pi sqrt163) = 640320^3+744", "identity", dict(lhs="exp(pi*sqrt(163))", rhs="640320**3+744"), "reject"),
    ("A12 e^pi - pi = 20", "identity", dict(lhs="exp(pi)-pi", rhs="20"), "reject"),
    ("A13 (x+y)^2 = x^2+y^2", "identity", dict(lhs="(x+y)**2", rhs="x**2+y**2"), "reject"),
    ("A14 sin(x+y) = sin x + sin y", "identity", dict(lhs="sin(x+y)", rhs="sin(x)+sin(y)"), "reject"),
    ("A15 sin^2 x = x^2 - x^4/3 (truncated series)", "identity", dict(lhs="sin(x)**2", rhs="x**2-x**4/3"), "reject"),
    ("A16 |sin x e^-x| > 0.35 on [0,10]  (sup .3224)", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.35"), "reject"),
    ("A17 |sin x e^-x| > 0.30 on [0,10]", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.30"), "accept"),
    ("A18 |sin x e^-x| > 0.3223 on [0,10]", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.3223"), "accept"),
    ("A19 |sin x e^-x| > 0.3224 on [0,10]", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.3224"), "reject"),
    ("A20 T = 2 pi sqrt(L/g)", "dimensions", dict(equation="T = 2*pi*sqrt(L/g)", dims={"T": "T", "L": "L", "g": "L T^-2"}), "accept"),
    ("A21 T = 2 pi sqrt(g/L)", "dimensions", dict(equation="T = 2*pi*sqrt(g/L)", dims={"T": "T", "L": "L", "g": "L T^-2"}), "reject"),
    ("A22 E = m v^n (unknown n)", "dimensions", dict(equation="En = m*v**n", dims={"En": "M L^2 T^-2", "m": "M", "v": "L T^-1"}), "conditional"),
    ("A23 E = h f", "dimensions", dict(equation="En = h*f", dims={"En": "M L^2 T^-2", "h": "M L^2 T^-1", "f": "T^-1"}), "accept"),
    ("A24 e^pi - pi ~ 20 is meaningful", "coincidence", dict(a=19.99909997, b=20.0, formula="exp(pi)-pi"), "reject"),
    ("A25 6 pi^5 ~ proton/electron mass ratio", "coincidence", dict(a=1836.118108, b=1836.15267343, formula="6*pi**5"), "reject"),
    ("A26 (x^2-1)/(x-1) = x+1", "identity", dict(lhs="(x**2-1)/(x-1)", rhs="x+1"), "accept"),
    ("A27 x/x = 1", "identity", dict(lhs="x/x", rhs="1"), "accept"),
]

RED_B = [
    ("B1  sqrt(x^4) = x^2 (true for all reals)", "identity", dict(lhs="sqrt(x**4)", rhs="x**2"), "accept"),
    ("B2  sqrt(x^2) = |x| (true for all reals)", "identity", dict(lhs="sqrt(x**2)", rhs="Abs(x)"), "accept"),
    ("B3  log(x^3) = 3 log x, no domain", "identity", dict(lhs="log(x**3)", rhs="3*log(x)"), "conditional"),
    ("B4  log(x^3) = 3 log x, declared x>0", "identity", dict(lhs="log(x**3)", rhs="3*log(x)", domain=POS), "accept"),
    ("B5  y = y + 1e-70 y^3", "identity", dict(lhs="y", rhs="y + 10**(-70)*y**3"), "reject"),
    ("B6  e to 50 decimals (correct)", "value", dict(expr="exp(1)", claimed=digits("e", 50)), "accept"),
    ("B7  sqrt2 to 52 decimals (last digit -1)", "value", dict(expr="sqrt(2)", claimed=digits("sqrt2", 52, -1)), "reject"),
    ("B8  cos3x = 4c^3 - 3c", "identity", dict(lhs="cos(3*x)", rhs="4*cos(x)**3-3*cos(x)"), "accept"),
    ("B9  sin^4 - cos^4 = -cos2x", "identity", dict(lhs="sin(x)**4-cos(x)**4", rhs="-cos(2*x)"), "accept"),
    ("B10 sin5x = 16s^5-20s^3+5s", "identity", dict(lhs="sin(5*x)", rhs="16*sin(x)**5-20*sin(x)**3+5*sin(x)"), "accept"),
    ("B11 e^(pi sqrt67) = 5280^3+744", "identity", dict(lhs="exp(pi*sqrt(67))", rhs="5280**3+744"), "reject"),
    ("B12 pi^4 + pi^5 = e^6", "identity", dict(lhs="pi**4+pi**5", rhs="exp(6)"), "reject"),
    ("B13 (x+y)^3 = x^3+y^3", "identity", dict(lhs="(x+y)**3", rhs="x**3+y**3"), "reject"),
    ("B14 cos(x+y) = cos x + cos y", "identity", dict(lhs="cos(x+y)", rhs="cos(x)+cos(y)"), "reject"),
    ("B15 cos^2 x = 1 - x^2 + x^4/3 (truncated)", "identity", dict(lhs="cos(x)**2", rhs="1-x**2+x**4/3"), "reject"),
    ("B16 |cos x e^(-x/2)| > 0.99 on [0,10]", "exceeds", dict(expr="cos(x)*exp(-x/2)", var="x", lo=0, hi=10, c="0.99"), "accept"),
    ("B17 |cos x e^(-x/2)| > 1.01 on [0,10]", "exceeds", dict(expr="cos(x)*exp(-x/2)", var="x", lo=0, hi=10, c="1.01"), "reject"),
    ("B18 |x e^-x| > 0.3678 on [0,10]  (sup 1/e)", "exceeds", dict(expr="x*exp(-x)", var="x", lo=0, hi=10, c="0.3678"), "accept"),
    ("B19 |x e^-x| > 0.3680 on [0,10]", "exceeds", dict(expr="x*exp(-x)", var="x", lo=0, hi=10, c="0.3680"), "reject"),
    ("B20 v = sqrt(2 g h)", "dimensions", dict(equation="v = sqrt(2*g*h)", dims={"v": "L T^-1", "g": "L T^-2", "h": "L"}), "accept"),
    ("B21 v = sqrt(g/h)", "dimensions", dict(equation="v = sqrt(g/h)", dims={"v": "L T^-1", "g": "L T^-2", "h": "L"}), "reject"),
    ("B22 F = m v^n / t (unknown n)", "dimensions", dict(equation="F = m*v**n/t", dims={"F": "M L T^-2", "m": "M", "v": "L T^-1", "t": "T"}), "conditional"),
    ("B23 E = k x^2 / 2 (spring)", "dimensions", dict(equation="En = k*x**2/2", dims={"En": "M L^2 T^-2", "k": "M T^-2", "x": "L"}), "accept"),
    ("B24 2^10 ~ 10^3 is meaningful", "coincidence", dict(a=1024.0, b=1000.0, formula="2**10"), "reject"),
    ("B25 pi^2 ~ g = 9.80665 is meaningful", "coincidence", dict(a=9.8696044, b=9.80665, formula="pi**2"), "reject"),
    ("B26 (x^3-1)/(x-1) = x^2+x+1", "identity", dict(lhs="(x**3-1)/(x-1)", rhs="x**2+x+1"), "accept"),
    ("B27 x (1/x) = 1", "identity", dict(lhs="x*(1/x)", rhs="1"), "accept"),
]


RED_C = [
    ("C1  sqrt(y^2) = y, no domain", "identity", dict(lhs="sqrt(y**2)", rhs="y"), "conditional"),
    ("C2  sqrt(y^2) = y, declared y>0", "identity", dict(lhs="sqrt(y**2)", rhs="y", domain=POS), "accept"),
    ("C3  log(y^4) = 4 log y, no domain", "identity", dict(lhs="log(y**4)", rhs="4*log(y)"), "conditional"),
    ("C4  log(xy) = log x + log y, declared positive", "identity", dict(lhs="log(x*y)", rhs="log(x)+log(y)", domain=POS), "accept"),
    ("C5  log(xy) = log x + log y, no domain", "identity", dict(lhs="log(x*y)", rhs="log(x)+log(y)"), "conditional"),
    ("C6  x = x + 1e-90 x^3", "identity", dict(lhs="x", rhs="x + 10**(-90)*x**3"), "reject"),
    ("C7  pi to 40 decimals (correct)", "value", dict(expr="pi", claimed=digits("pi", 40)), "accept"),
    ("C8  e to 45 decimals (last digit +1)", "value", dict(expr="exp(1)", claimed=digits("e", 45, +1)), "reject"),
    ("C9  cos4x = 8c^4 - 8c^2 + 1", "identity", dict(lhs="cos(4*x)", rhs="8*cos(x)**4-8*cos(x)**2+1"), "accept"),
    ("C10 sin2x = 2 sinx cosx", "identity", dict(lhs="sin(2*x)", rhs="2*sin(x)*cos(x)"), "accept"),
    ("C11 cos^6 + sin^6 = 1 - 3 sin^2 cos^2", "identity", dict(lhs="cos(x)**6+sin(x)**6", rhs="1-3*sin(x)**2*cos(x)**2"), "accept"),
    ("C12 e^(pi sqrt43) = 960^3+744", "identity", dict(lhs="exp(pi*sqrt(43))", rhs="960**3+744"), "reject"),
    ("C13 e^(pi sqrt19) = 96^3+744", "identity", dict(lhs="exp(pi*sqrt(19))", rhs="96**3+744"), "reject"),
    ("C14 (x+y)^4 = x^4+y^4", "identity", dict(lhs="(x+y)**4", rhs="x**4+y**4"), "reject"),
    ("C15 cos2x = cos^2 x", "identity", dict(lhs="cos(2*x)", rhs="cos(x)**2"), "reject"),
    ("C16 |sin x e^(-x/3)| > 0.624 on [0,12]", "exceeds", dict(expr="sin(x)*exp(-x/3)", var="x", lo=0, hi=12, c="0.624"), "accept"),
    ("C17 |sin x e^(-x/3)| > 0.627 on [0,12]", "exceeds", dict(expr="sin(x)*exp(-x/3)", var="x", lo=0, hi=12, c="0.627"), "reject"),
    ("C18 |x e^(-2x)| > 0.18 on [0,5]  (sup .1839)", "exceeds", dict(expr="x*exp(-2*x)", var="x", lo=0, hi=5, c="0.18"), "accept"),
    ("C19 |x e^(-2x)| > 0.19 on [0,5]", "exceeds", dict(expr="x*exp(-2*x)", var="x", lo=0, hi=5, c="0.19"), "reject"),
    ("C20 E = m g h", "dimensions", dict(equation="En = m*g*h", dims={"En": "M L^2 T^-2", "m": "M", "g": "L T^-2", "h": "L"}), "accept"),
    ("C21 E = m g t", "dimensions", dict(equation="En = m*g*t", dims={"En": "M L^2 T^-2", "m": "M", "g": "L T^-2", "t": "T"}), "reject"),
    ("C22 p = m v^n (unknown n)", "dimensions", dict(equation="p = m*v**n", dims={"p": "M L T^-1", "m": "M", "v": "L T^-1"}), "conditional"),
    ("C23 a = v^2 / r", "dimensions", dict(equation="a = v**2/r", dims={"a": "L T^-2", "v": "L T^-1", "r": "L"}), "accept"),
    ("C24 a = v / r", "dimensions", dict(equation="a = v/r", dims={"a": "L T^-2", "v": "L T^-1", "r": "L"}), "reject"),
    ("C25 phi ~ 8/5 is meaningful", "coincidence", dict(a=1.6, b=1.6180339887, formula="8/5"), "reject"),
    ("C26 sqrt(10) ~ pi is meaningful", "coincidence", dict(a=3.16227766, b=3.14159265, formula="sqrt(10)"), "reject"),
    ("C27 x^2 - 4 = (x-2)(x+2)", "identity", dict(lhs="x**2-4", rhs="(x-2)*(x+2)"), "accept"),
]


def classify(v) -> str:
    if v.accepted:
        return "accept"
    return "conditional" if v.status == "CONDITIONAL" else "reject"


def run_set(name, items, eng):
    ok = 0
    print(f"\n=== red-team set {name} ===")
    for label, kind, kw, expected in items:
        v = getattr(eng, kind)(**kw)
        got = classify(v)
        good = got == expected
        ok += good
        print(f"{'ok ' if good else 'BAD'} {label:<46} expected {expected:<11} got {v.status:<12} {'' if good else '<- ' + v.detail[:95]}")
    print(f"set {name}: {ok} of {len(items)} correct")
    return ok, len(items)


if __name__ == "__main__":
    eng = NKPMath(seed=1)
    a = run_set("A", RED_A, eng)
    b = run_set("B", RED_B, eng)
    c = run_set("C", RED_C, eng)
    print(f"\nA {a[0]}/{a[1]} (used for fixing)   B {b[0]}/{b[1]} (held out, but seen once)   C {c[0]}/{c[1]} (fresh, written after the fixes)")


In [ ]:
%%writefile nkp_math_bench.py
"""
Benchmark for NKP-Math.  Two parts, with different meanings:

PART 1  Synthetic claim zoo (labels known BY CONSTRUCTION; each seed = 130 claims)
   identities (true templates; false by coefficient/trig swaps, tiny 1e-2..1e-25 perturbations,
   and decimal "shortfall" arithmetic), decimal value claims (digit-level errors),
   existence bounds (narrow Gaussian peaks), and unit-consistency equations.
   Baselines: Trust (accept all) | Lookup (text match against a textbook list) |
              Float1 (one random point, float64, tol 1e-9) | Float5 (five points).
   This part measures CAPABILITY COVERAGE: which failure modes does each approach catch?  It is not a
   contest on a shared hard task, and the generator and engine were written by the same person.

PART 2  Real claims taken from the user's own documents (ground truth from independent calculations
   done earlier in the project).  This is the meaningful part.

PRE-REGISTERED CRITERIA (written before any run):
  M1  NKP-Math false-accept rate <= 1% overall
  M2  overall accuracy beats the best baseline by > 2 SE (paired over seeds)
  M3  in no family is NKP-Math worse than the best baseline by > 2 SE
  M4  every real case receives the expected verdict
Dev seeds 0-9; final test = fresh seeds 100-119, run once.

    python nkp_math_bench.py --real
    python nkp_math_bench.py --seeds 0:10 --out dev.json ;  python nkp_math_bench.py --report dev.json
"""
from __future__ import annotations

import json
import math
import random
import re
import statistics
import sys
from decimal import ROUND_HALF_UP, Decimal, getcontext

import mpmath as mp
import numpy as np
import sympy as sp

from nkp_math import NKPMath, parse

getcontext().prec = 90
FAMILIES = ["identity", "shortfall", "value", "bound", "dimensions"]
METHODS = ["Trust", "Lookup", "Float1", "Float5", "NKP-Math"]


def fib(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


# ------------------------------------------------------------------ generators
def _true_identity(rng):
    k = rng.randrange(9); a = rng.randint(2, 6)
    return [
        (f"sin({a}*x)**2+cos({a}*x)**2", "1"),
        (f"(x+{a})**3", f"x**3+{3*a}*x**2+{3*a*a}*x+{a**3}"),
        (f"x**2-{a*a}", f"(x-{a})*(x+{a})"),
        (f"cos({2*a}*x)", f"1-2*sin({a}*x)**2"),
        ("sin(x+y)", "sin(x)*cos(y)+cos(x)*sin(y)"),
        (f"exp({a}*x+y)", f"exp(x)**{a}*exp(y)"),
        (f"log(x**{a}*y)", f"{a}*log(x)+log(y)"),
        (f"x**{a}-1", "(x-1)*(" + "+".join(f"x**{j}" for j in range(a)) + ")"),
        (f"phi**{a+2}", f"{fib(a+2)}*phi+{fib(a+1)}"),
    ][k]


def _oracle_identity(L, R):
    """Ground truth for mutated identities, computed independently of the engine (80 digits, 24 points)."""
    r = random.Random(7)
    d = parse(L) - parse(R)
    syms = sorted(d.free_symbols, key=str)
    f = sp.lambdify(syms, d, modules="mpmath")
    mp.mp.dps = 80
    try:
        for _ in range(24):
            pt = [mp.mpf(0.1) + mp.mpf(2.9) * mp.mpf(r.getrandbits(250)) / mp.mpf(2) ** 250 for _ in syms]
            if abs(f(*pt)) > mp.mpf(10) ** -60:
                return False
    finally:
        mp.mp.dps = 50
    return True


def gen_identity(rng):
    L, R = _true_identity(rng)
    dom = {"x": (0.1, 3.0), "y": (0.1, 3.0)} if "log" in L else None       # log identities are claims about positive inputs
    if rng.random() < 0.5:
        return dict(family="identity", L=L, R=R, label=True, domain=dom)
    mode = rng.choice(["perturb", "perturb", "coef", "swap"])
    if mode == "perturb":
        R2 = f"({R})+10**(-{rng.choice([2, 4, 6, 9, 12, 16, 20, 25])})*x**2"
    elif mode == "coef":
        nums = [m for m in re.finditer(r"(?<![\w.*^])\d+(?![\w.])", R)] or [m for m in re.finditer(r"\d+", R)]
        if nums:
            m = rng.choice(nums); R2 = R[:m.start()] + str(int(m.group()) + 1) + R[m.end():]
        else:
            R2 = f"({R})+x"                                           # no numerals to alter: add a term instead
    else:
        R2 = R.replace("sin", "cos", 1) if "sin" in R else R.replace("exp", "exp(1)*exp", 1) if "exp" in R else f"({R})*2"
    return dict(family="identity", L=L, R=R2, label=_oracle_identity(L, R2), domain=dom)


def gen_shortfall(rng):
    n = rng.randint(3, 25)
    if rng.random() < 0.5:
        return dict(family="shortfall", L=f"3*0.{'3'*n}", R="1", label=False)                     # 3 x 0.33..3 = 1 ?
    return dict(family="shortfall", L=f"3*(10**{n}-1)/(3*10**{n})+10**(-{n})", R="1", label=True)


_VALS = ["pi", "phi", "sqrt(2)", "sqrt(3)", "log(2)", "exp(pi)", "2*pi/phi**2", "pi**2/6", "sqrt(5)", "exp(1)"]


def gen_value(rng):
    e = rng.choice(_VALS); k = rng.randint(3, 28)
    v = Decimal(str(sp.N(parse(e), 80)))
    q = v.quantize(Decimal(1).scaleb(-k), rounding=ROUND_HALF_UP)
    if rng.random() < 0.5:
        return dict(family="value", expr=e, claimed=format(q, "f"), label=True)
    bad = q + rng.choice([-1, 1]) * Decimal(1).scaleb(-k)
    return dict(family="value", expr=e, claimed=format(bad, "f"), label=False)


def gen_bound(rng):
    a = round(rng.uniform(0.5, 5), 3); x0 = round(rng.uniform(1, 9), 4)
    w = rng.choice([0.002, 0.01, 0.05]); kk = rng.choice([1, 3]); eps = rng.choice([0.001, 0.01, 0.05])
    true = rng.random() < 0.5
    c = a * (1 - eps) if true else a * (1 + eps)
    return dict(family="bound", expr=f"{a}*exp(-((x-{x0})/{w})**2)*cos({kk}*(x-{x0}))", c=format(Decimal(c).quantize(Decimal("1e-9")), "f"), label=true)


_QD = {"m": "M", "a": "L T^-2", "v": "L T^-1", "d": "L", "t": "T", "F": "M L T^-2", "En": "M L^2 T^-2", "p": "M L T^-1", "P": "M L^2 T^-3"}
_TRUE_EQ = ["F = m*a", "En = m*v**2/2", "p = m*v", "v = d/t", "P = F*v", "En = F*d", "a = v/t", "En = P*t", "d = v*t", "F = p/t"]
_BAD_EQ = ["En = m*v", "F = m*v", "p = m*v**2", "v = d*t", "P = F*d", "En = m*v**2 + m*v", "a = v*t", "En = F*t", "d = v/t", "F = m*sin(d)", "En = m*exp(t)", "P = F*v + m*a"]


def gen_dims(rng):
    true = rng.random() < 0.5
    eq = rng.choice(_TRUE_EQ if true else _BAD_EQ)
    if rng.random() < 0.5:                                        # harmless decoration: a dimensionless factor
        lhs, rhs = eq.split("="); eq = f"{lhs.strip()} = {rng.choice(['2', '3', 'pi', '1/2'])}*({rhs.strip()})"
        if not true and "sin" in eq or not true and "exp" in eq: pass
    return dict(family="dimensions", eq=eq, label=true)


def make_seed(seed):
    rng = random.Random(seed)
    out = [gen_identity(rng) for _ in range(40)] + [gen_shortfall(rng) for _ in range(10)] + [gen_value(rng) for _ in range(30)]
    out += [gen_bound(rng) for _ in range(20)] + [gen_dims(rng) for _ in range(30)]
    return out


# ------------------------------------------------------------------ baselines and engine
def fparse(s):
    return sp.parse_expr(s.replace("^", "**"), local_dict={"phi": sp.GoldenRatio, "pi": sp.pi, "euler": sp.E})


_LOOKUP = {("sin(2*x)**2+cos(2*x)**2", "1"), ("sin(x+y)", "sin(x)*cos(y)+cos(x)*sin(y)"), ("x**2-4", "(x-2)*(x+2)"),
           ("exp(2*x+y)", "exp(x)**2*exp(y)"), ("phi**4", "3*phi+2"), ("(x+2)**3", "x**3+6*x**2+12*x+8")}
_LOOKUP_VALUES = {("pi", "3.14159265358979"), ("phi", "1.6180339887"), ("sqrt(2)", "1.41421356237"), ("exp(1)", "2.71828182845")}
_LOOKUP_EQ = {"F = m*a", "En = m*v**2/2", "p = m*v", "v = d/t"}


def float_check(c, npts, rng):
    f = c["family"]
    if f in ("identity", "shortfall"):
        d = fparse(c["L"]) - fparse(c["R"])
        syms = sorted(d.free_symbols, key=str)
        fn = sp.lambdify(syms, d, "numpy")
        fl = sp.lambdify(syms, fparse(c["L"]), "numpy")
        for _ in range(npts):
            pt = [rng.uniform(0.1, 3.0) for _ in syms]
            if abs(float(fn(*pt))) > 1e-9 * max(1.0, abs(float(fl(*pt)))):
                return False
        return True
    if f == "value":
        return abs(float(sp.N(fparse(c["expr"]), 15)) - float(c["claimed"])) <= 1e-9
    if f == "bound":
        fn = sp.lambdify(sp.Symbol("x"), fparse(c["expr"]), "numpy")
        xs = np.linspace(0, 10, 400 if npts > 1 else 50)
        return bool(np.max(np.abs(fn(xs))) > float(c["c"]))
    return False                                                  # numeric checks cannot see units


def decide(method, c, eng, rng):
    f = c["family"]
    if method == "Trust":
        return True
    if method == "Lookup":
        if f in ("identity", "shortfall"): return (c["L"], c["R"]) in _LOOKUP
        if f == "value": return (c["expr"], c["claimed"]) in _LOOKUP_VALUES
        if f == "dimensions": return c["eq"] in _LOOKUP_EQ
        return False
    if method == "Float1": return float_check(c, 1, rng)
    if method == "Float5": return float_check(c, 5, rng)
    if f in ("identity", "shortfall"): return eng.identity(c["L"], c["R"], domain=c.get("domain")).accepted
    if f == "value": return eng.value(c["expr"], c["claimed"]).accepted
    if f == "bound": return eng.exceeds(c["expr"], "x", 0, 10, c["c"]).status == "PROVED"
    return eng.dimensions(c["eq"], _QD).accepted


def run_seed(seed):
    claims = make_seed(seed); eng = NKPMath(seed=seed); rng = random.Random(seed + 1)
    tally = {m: {fam: [0, 0, 0, 0] for fam in FAMILIES} for m in METHODS}            # correct, total, false-accepts, false-total
    for c in claims:
        for m in METHODS:
            d = decide(m, c, eng, rng)
            t = tally[m][c["family"]]
            t[0] += int(d == c["label"]); t[1] += 1
            if not c["label"]:
                t[3] += 1; t[2] += int(d)
    return tally


# ------------------------------------------------------------------ part 2: real claims from the user's documents
REAL = [
    # (kind, args, expected, source)
    ("identity", ("2*cos(pi/5)", "phi"), "accept", "screenshot:gemini-identity"),
    ("identity", ("phi", "1+1/phi"), "accept", "screenshot:gemini-identity"),
    ("identity", ("phi**2", "phi+1"), "accept", "doc:copilot-parameters"),
    ("identity", ("3*0.3333", "1"), "reject", "screenshot:triadic-flaw"),
    ("identity", ("0.3333+0.3333+0.3333", "0.9999"), "accept", "user:own-arithmetic"),
    ("identity", ("0.3333+0.3333+0.3333+0.0001", "1"), "accept", "user:own-arithmetic"),
    ("identity", ("1-3*(10**8-1)/(3*10**8)", "10**(-8)"), "accept", "screenshot:triadic-flaw"),
    ("value", ("2*pi/phi**2", "2.3999632297"), "accept", "doc:sieve-script"),
    ("value", ("3**9", "19683"), "accept", "screenshot:gemini-tictactoe"),
    ("exceeds", ("sin(x*pi)*phi**(x/10)/(pi*phi)", "x", 0, 10, "0.3333333333333333"), "reject", "doc:gate-script"),
    ("dimensions", ("Gmn = 8*pi*G/c**4*Tmn", {"Gmn": "L^-2", "G": "L^3 M^-1 T^-2", "c": "L T^-1", "Tmn": "M L^-1 T^-2"}), "accept", "gr-baseline"),
    ("dimensions", ("Gmn = 8*pi*G/c**4*(I*sqrt(X))**(1+b)*Tmn/sqrt(X)", {"Gmn": "L^-2", "G": "L^3 M^-1 T^-2", "c": "L T^-1", "Tmn": "M L^-1 T^-2", "X": "M L^-1 T^-2"}), "conditional", "screenshot:copilot-single-equation"),
    ("dimensions", ("Pw = al*Cc*f*V**2", {"Pw": "M L^2 T^-3", "Cc": "Q^2 T^2 M^-1 L^-2", "f": "T^-1", "V": "M L^2 T^-2 Q^-1"}), "accept", "screenshot:google-ai-cpu"),
    ("dimensions", ("Pj = Ic**2*Rr", {"Pj": "M L^2 T^-3", "Ic": "Q T^-1", "Rr": "M L^2 T^-1 Q^-2"}), "accept", "screenshot:google-ai-cpu"),
    ("dimensions", ("Ic = Isat*exp(V)", {"Ic": "Q T^-1", "Isat": "Q T^-1", "V": "M L^2 T^-2 Q^-1"}), "reject", "screenshot:google-ai-cpu"),
    ("dimensions", ("Gmn = rho", {"Gmn": "L^-2"}), "reject", "doc:unified-equation"),
    ("coincidence", (360 * (1 - 1 / ((1 + 5 ** 0.5) / 2)), 137.035999084, "360*(1-1/phi)"), "reject", "user:hydrogen-similarity"),
]


def run_real():
    eng = NKPMath(seed=0); ok = 0
    print(f"{'#':>2} {'source':<34}{'verdict':<13}{'expected':<12} claim / detail")
    for i, (kind, args, exp, src) in enumerate(REAL, 1):
        v = getattr(eng, kind)(*args, source=src)
        got = "accept" if v.accepted else "reject" if v.status in ("REFUTED", "ILL_FORMED", "COINCIDENCE") else "conditional"
        good = got == exp; ok += good
        name = args[0] if isinstance(args[0], str) else f"{args[0]:.4f} vs {args[1]}"
        print(f"{i:>2} {src:<34}{v.status:<13}{exp:<12} {'OK ' if good else 'BAD'} {str(name)[:46]} | {v.detail[:90]}")
    print(f"\nM4: {ok} of {len(REAL)} real claims received the expected verdict -> {'PASS' if ok == len(REAL) else 'FAIL'}")
    print("\nsource track record (true / false / open -> reliability):")
    for s, t, f, o, r in eng.ledger.table():
        print(f"  {s:<34}{t} / {f} / {o}  -> {r:.2f}")
    return ok == len(REAL)


# ------------------------------------------------------------------ reporting
def se(x):
    return statistics.stdev(x) / math.sqrt(len(x)) if len(x) > 1 else 0.0


def report(files):
    runs = []
    for fn in files: runs += json.load(open(fn))
    n = len(runs)

    def acc(r, m, fams):
        c = sum(r[m][f][0] for f in fams); t = sum(r[m][f][1] for f in fams); return c / t

    def fa(r, m):
        a = sum(r[m][f][2] for f in FAMILIES); b = sum(r[m][f][3] for f in FAMILIES); return a / b
    print(f"{n} seeds, {n*130} claims.   accuracy by family (mean over seeds)")
    print(f"{'family':<12}" + "".join(f"{m:>11}" for m in METHODS))
    for fam in FAMILIES + ["ALL"]:
        fams = FAMILIES if fam == "ALL" else [fam]
        print(f"{fam:<12}" + "".join(f"{statistics.mean(acc(r, m, fams) for r in runs):>11.3f}" for m in METHODS))
    print(f"{'false-accept':<12}" + "".join(f"{statistics.mean(fa(r, m) for r in runs):>11.3f}" for m in METHODS))
    best = lambda fams: max(METHODS[:-1], key=lambda m: statistics.mean(acc(r, m, fams) for r in runs))
    print("\nPRE-REGISTERED CRITERIA")
    m1 = statistics.mean(fa(r, "NKP-Math") for r in runs)
    print(f"  M1 false-accept rate {m1:.3f}  {'PASS' if m1 <= 0.01 else 'FAIL'}")
    b = best(FAMILIES); d = [acc(r, "NKP-Math", FAMILIES) - acc(r, b, FAMILIES) for r in runs]
    print(f"  M2 overall vs best baseline ({b}): {statistics.mean(d):+.3f}±{se(d):.3f}  {'PASS' if statistics.mean(d) - 2 * se(d) > 0 else 'FAIL'}")
    ok3 = True
    for fam in FAMILIES:
        b = best([fam]); d = [acc(r, "NKP-Math", [fam]) - acc(r, b, [fam]) for r in runs]
        good = statistics.mean(d) + 2 * se(d) >= 0; ok3 &= good
        print(f"  M3 {fam:<11} vs best baseline ({b}): {statistics.mean(d):+.3f}±{se(d):.3f}  {'ok' if good else 'WORSE'}")
    print(f"  M3 overall: {'PASS' if ok3 else 'FAIL'}")


def _arg(a, flag):
    """Value after a flag.  A missing or mistyped flag STOPS the run; there is no silent default."""
    if flag not in a or a.index(flag) + 1 >= len(a):
        sys.exit(f"ERROR: {flag} is required.  Usage:\n"
                 "  python nkp_math_bench.py --seeds 0:10 --out dev.json\n"
                 "  python nkp_math_bench.py --seeds 100:120 --out final.json --final\n"
                 "  python nkp_math_bench.py --report dev.json\n  python nkp_math_bench.py --real")
    return a[a.index(flag) + 1]


if __name__ == "__main__":
    a = sys.argv
    known = {"--real", "--report", "--seeds", "--out", "--final"}
    bad = [x for x in a[1:] if x.startswith("-") and x not in known]
    if bad:
        sys.exit(f"ERROR: unknown option {bad}.  Did you mistype --seeds / --out / --final / --real / --report?")
    if "--real" in a:
        run_real()
    elif "--report" in a:
        report(a[a.index("--report") + 1:])
    else:
        lo, hi = map(int, _arg(a, "--seeds").split(":"))
        out = _arg(a, "--out")
        final = "--final" in a
        if final and lo < 100:
            sys.exit(f"ERROR: --final means fresh seeds 100 and up, but you gave {lo}:{hi} (dev data).")
        if not final and hi > 100:
            sys.exit(f"ERROR: seeds {lo}:{hi} reach the final-test range (100+). Add --final if this is the one-time final run.")
        print(f"{'FINAL TEST' if final else 'DEV RUN'}: seeds {lo}:{hi}")
        res = [run_seed(s) for s in range(lo, hi)]
        json.dump(res, open(out, "w"))
        print(f"seeds {lo}:{hi} done -> {out}")


In [ ]:
%%writefile nkp_check.py
"""
nkp_check.py - one-file referee gate for NKP-Math.  Run it after EVERY AI edit.

    python nkp_check.py

It never trusts silence.  Each check must print PASS or the whole run ends in FAIL (exit code 1).
Put it in the same folder as nkp_math.py and nkp_math_redteam.py.  Needs sympy and mpmath.

Checks
  1  nkp_math.py still contains its test suite (counted from the code, not from the message it prints)
  2  the test suite runs, prints something, and passes
  3  red-team sets A, B, C all score full marks
  4  zero false accepts (a claim labelled reject/conditional that the engine accepted)
  5  the label-correction note and the A/B annotations are still in the red-team file
  6  if nkp_math_bench.py is present: no literal 'fam' label bug, and no silent seed default
  6c if nkp_ledger.py is present: its tests run and pass
  7  fingerprints of the files, so you can see when something changed
"""
from __future__ import annotations

import ast
import contextlib
import hashlib
import io
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

MIN_ASSERTS = 28          # the referee had 28 checks; fewer means something was deleted
failures = []


def report(name: str, ok: bool, note: str = "") -> None:
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f"  ({note})" if note else ""))
    if not ok:
        failures.append(name)


def read(fn: str) -> str:
    with open(os.path.join(HERE, fn), encoding="utf-8") as f:
        return f.read()


def main() -> int:
    try:
        math_src = read("nkp_math.py")
        red_src = read("nkp_math_redteam.py")
    except FileNotFoundError as e:
        print(f"FAIL  missing file: {e.filename}")
        return 1

    # 1. the tests still exist (the printed "28" is just a literal, so count real assert statements)
    tree = ast.parse(math_src)
    fn = next((n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef) and n.name == "run_tests"), None)
    n_assert = sum(isinstance(n, ast.Assert) for n in ast.walk(fn)) if fn else 0
    report("test suite present in nkp_math.py", n_assert >= MIN_ASSERTS, f"{n_assert} asserts, need >= {MIN_ASSERTS}")
    report("command-line block present (--test)", '"--test"' in math_src or "'--test'" in math_src)

    # 2. run the tests; empty output counts as failure
    import nkp_math
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            nkp_math.run_tests()
        out = buf.getvalue().strip()
        report("tests ran and passed", bool(out), out if out else "EMPTY OUTPUT = failure")
    except Exception as e:  # AssertionError or anything else
        report("tests ran and passed", False, f"{type(e).__name__}: {e}"[:120])

    # 3 + 4. red-team sets
    import nkp_math_redteam as rt
    eng = nkp_math.NKPMath(seed=1)
    false_accepts = []
    for name, items in (("A", rt.RED_A), ("B", rt.RED_B), ("C", rt.RED_C)):
        ok = 0
        for label, kind, kw, expected in items:
            v = getattr(eng, kind)(**kw)
            got = rt.classify(v)
            ok += got == expected
            if got == "accept" and expected != "accept":
                false_accepts.append(label)
        report(f"red-team set {name}", ok == len(items), f"{ok}/{len(items)}")
    report("zero false accepts", not false_accepts, "; ".join(false_accepts))

    # 5. the honesty record
    report("label-correction note kept", "LABEL CORRECTION" in red_src)
    report("A-fix / B-held-out annotations kept", "used for fixing" in red_src and "held out" in red_src)

    # 6. benchmark guards (only if the file is there)
    bench_src = read("nkp_math_bench.py") if os.path.exists(os.path.join(HERE, "nkp_math_bench.py")) else ""
    if bench_src:
        report("benchmark: family label bug absent", "{'fam'" not in bench_src and '{"fam"' not in bench_src)
        report("benchmark: --seeds required, --final guard present",
               'sys.exit' in bench_src and '"--final"' in bench_src and "or '0:10'" not in bench_src and 'else "0:10"' not in bench_src)

    # 6c. the claims ledger (only if the file is there); empty output counts as failure
    led_path = os.path.join(HERE, "nkp_ledger.py")
    led_src = read("nkp_ledger.py") if os.path.exists(led_path) else ""
    if led_src:
        import nkp_ledger
        lbuf = io.StringIO()
        try:
            with contextlib.redirect_stdout(lbuf):
                nkp_ledger.run_tests()
            lout = lbuf.getvalue().strip()
            report("ledger tests ran and passed", bool(lout), lout if lout else "EMPTY OUTPUT = failure")
        except Exception as e:
            report("ledger tests ran and passed", False, f"{type(e).__name__}: {e}"[:120])

    # 6d. the real-claims test harness (only if the file is there)
    rt_path = os.path.join(HERE, "nkp_realtest.py")
    rt_src = read("nkp_realtest.py") if os.path.exists(rt_path) else ""
    if rt_src:
        import nkp_realtest
        rbuf = io.StringIO()
        try:
            with contextlib.redirect_stdout(rbuf):
                nkp_realtest.run_tests()
            rout = rbuf.getvalue().strip().splitlines()
            report("real-claims harness tests ran and passed", bool(rout) and rout[-1].startswith("all "), rout[-1] if rout else "EMPTY OUTPUT = failure")
        except Exception as e:
            report("real-claims harness tests ran and passed", False, f"{type(e).__name__}: {e}"[:120])

    # 7. fingerprints
    srcs = [("nkp_math.py", math_src), ("nkp_math_redteam.py", red_src)] + ([("nkp_math_bench.py", bench_src)] if bench_src else []) + ([("nkp_ledger.py", led_src)] if led_src else []) + ([("nkp_realtest.py", rt_src)] if rt_src else [])
    for fname, src in srcs:
        print(f"      {fname:<22} sha256 {hashlib.sha256(src.encode()).hexdigest()[:12]}")

    print("\nRESULT:", "ALL PASS" if not failures else f"FAIL ({len(failures)}): " + ", ".join(failures))
    return 0 if not failures else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile sieve_lab.py
"""
sieve_lab.py - audit and compare the Pi-Phi "1/3 sieve" as a token pre-filter.  One file, numpy only (no torch).

    python sieve_lab.py --audit     facts about the original sieve (share by window, period, aliasing)
    python sieve_lab.py --compare   original vs fixed vs stride vs random vs hybrid (synthetic, see caveats)
    python sieve_lab.py --test      self-checks; prints "all N checks passed" (empty output = failure)

Variants  (n = 1, 2, 3, ... token position)
  original  floor(n * 2*pi/phi^2) mod 3          (channel 0 = keep, 1 = compress, 2 = prune)
  fixed     floor(3 * frac(n / phi))             (same three channels, golden-ratio rotation in TURNS)
Helper API
  sieve_channels(n, variant)         -> int array of channels 0/1/2, length n
  sieve_select(x, variant)           -> survivors, position_ids, channels   (x: [seq, d] or [batch, seq, d])
  hybrid_keep(scores, rate, floor_share) -> bool mask: sieve coverage floor + highest-scored extra tokens
"""
from __future__ import annotations

import sys

import numpy as np

PHI = (1 + 5 ** 0.5) / 2
GOLDEN_ANGLE = 2 * np.pi / PHI ** 2          # 2.39996..., radians


# ------------------------------------------------------------------ the sieve
def sieve_channels(n: int, variant: str = "fixed", start: int = 1) -> np.ndarray:
    pos = np.arange(start, start + n, dtype=np.float64)
    if variant == "original":
        return np.floor(pos * GOLDEN_ANGLE).astype(np.int64) % 3
    if variant == "fixed":
        return np.floor(3 * ((pos / PHI) % 1.0)).astype(np.int64)
    raise ValueError(f"unknown variant {variant!r}")


def sieve_select(x: np.ndarray, variant: str = "fixed"):
    """Keep channel-0 tokens.  Returns (survivors, position_ids, channels).  Position ids are returned so a
    downstream model can still tell WHERE each survivor came from (the original dropped this information)."""
    seq_axis = x.ndim - 2
    ch = sieve_channels(x.shape[seq_axis], variant)
    keep = ch == 0
    pos_ids = np.flatnonzero(keep)
    return np.take(x, pos_ids, axis=seq_axis), pos_ids, ch


def hybrid_keep(scores: np.ndarray, rate: float, floor_share: float = 0.5) -> np.ndarray:
    """Coverage floor from the sieve + best-scoring extra tokens, within ONE total budget of rate * n.
    The floor (position-only, causal, content-blind) takes floor_share of the budget; a content scorer fills the
    rest.  The floor uses the fixed golden-ratio rotation with finer bins (keep share = rate * floor_share).
    scores: higher = more important."""
    n = len(scores)
    bins = max(int(round(1.0 / (rate * floor_share))), 1)
    pos = np.arange(1, n + 1, dtype=np.float64)
    keep = np.floor(bins * ((pos / PHI) % 1.0)).astype(np.int64) == 0
    extra = int(round(rate * n)) - int(keep.sum())
    if extra > 0:
        s = np.where(keep, -np.inf, scores)
        keep[np.argsort(-s)[:extra]] = True
    return keep


# ------------------------------------------------------------------ comparators
def stride_keep(n: int, k: int, offset: int = 0) -> np.ndarray:
    return (np.arange(n) % k) == offset


def random_keep(n: int, rate: float, rng) -> np.ndarray:
    m = np.zeros(n, bool); m[rng.choice(n, int(round(rate * n)), replace=False)] = True
    return m


# ------------------------------------------------------------------ metrics
def phase_retention(keep: np.ndarray, period: int) -> np.ndarray:
    """Share of tokens kept at each position-mod-period.  A fair filter keeps ~the same share at every phase."""
    pos = np.arange(1, len(keep) + 1)
    return np.array([keep[(pos % period) == p].mean() for p in range(period)])


def alias_score(keep: np.ndarray, periods=range(2, 13)) -> float:
    """Worst phase deviation from the overall keep rate over periods 2..12.  0 = no aliasing, ~0.8 = severe."""
    r = keep.mean()
    return float(max(np.abs(phase_retention(keep, p) - r).max() for p in periods))


def max_gap(keep: np.ndarray) -> int:
    idx = np.flatnonzero(keep)
    return int(np.diff(idx).max()) if len(idx) > 1 else len(keep)


# ------------------------------------------------------------------ audit
def audit() -> None:
    print(f"golden angle 2*pi/phi^2 = {GOLDEN_ANGLE:.10f}   divided by 3 = {GOLDEN_ANGLE / 3:.10f}")
    print(f"  that is within {0.8 - GOLDEN_ANGLE / 3:.2e} of 4/5, so the 'spiral' mod 3 is almost a period-5 pattern.")
    print(f"  the drift away from period 5 takes about {1 / (0.8 - GOLDEN_ANGLE / 3):,.0f} tokens.\n")
    ch = sieve_channels(12, "original")
    print("12-token demo routing:", ch.tolist(), " keep mask:", (ch == 0).astype(int).tolist(), "(matches the Colab output)\n")
    big = sieve_channels(1_000_000, "original")
    print(f"{'window':>14} {'keep':>7} {'compress':>9} {'prune':>7}   (original sieve; ideal is 0.333 each)")
    for a, b in [(0, 100), (0, 5000), (20000, 25000), (40000, 45000), (60000, 65000), (0, 1_000_000)]:
        w = big[a:b]
        print(f"{a:>7}:{b:<6} {(w == 0).mean():>7.3f} {(w == 1).mean():>9.3f} {(w == 2).mean():>7.3f}")
    k = sieve_channels(5000, "original") == 0
    print("\nkept positions, first 8:", (np.flatnonzero(k)[:8] + 1).tolist(), " -> every 5th token, same as stride 5")
    print("retention by position mod 5 (first 5000 tokens):", np.round(phase_retention(k, 5), 2).tolist())
    print("   (the sieve keeps 100% of one phase and 0% of the other four)")


# ------------------------------------------------------------------ compare
def compare(n: int = 4000, trials: int = 200, rate: float = 1 / 3) -> None:
    rng = np.random.default_rng(0)
    print(f"n={n} tokens, target keep rate {rate:.3f}.  Positional metrics are exact; the needle test is SYNTHETIC.\n")
    rows = {
        "original": sieve_channels(n, "original") == 0,
        "fixed": sieve_channels(n, "fixed") == 0,
        "stride-3": stride_keep(n, 3),
        "stride-5": stride_keep(n, 5, 4),
        "random": random_keep(n, rate, rng),
    }
    print(f"{'method':<10}{'keep':>7}{'max gap':>9}{'alias score':>13}")
    for k, m in rows.items():
        print(f"{k:<10}{m.mean():>7.3f}{max_gap(m):>9}{alias_score(m):>13.3f}")

    # synthetic needle test: 5% of tokens are 'important'; a content scorer sees them with noise
    print("\nneedle test: share of important tokens retained (mean of", trials, "trials)")
    res = {k: [] for k in ["original", "fixed", "stride-5", "random", "hybrid"]}
    for t in range(trials):
        important = rng.random(n) < 0.05
        scores = important * 1.0 + rng.normal(0, 0.5, n)                  # noisy scorer, honest but imperfect
        for k in ["original", "fixed", "stride-5", "random"]:
            m = rows[k] if k != "random" else random_keep(n, rate, rng)
            res[k].append(m[important].mean())
        res["hybrid"].append(hybrid_keep(scores, rate)[important].mean())
    for k, v in res.items():
        print(f"  {k:<14}{np.mean(v):.3f}")
    # worst case: important tokens sit at one phase mod 5 (tables, rows, repeated templates)
    print("\nworst case: every important token at position = 4 (mod 5)  (periodic structure)")
    imp = (np.arange(1, n + 1) % 5) == 4
    for k in ["original", "fixed", "stride-5", "stride-3"]:
        print(f"  {k:<10}{rows[k][imp].mean():.3f}")
    print("  (original and stride-5 keep either all of them or none, depending on phase)")


# ------------------------------------------------------------------ tests
def run_tests() -> None:
    n_ok = 0

    def check(cond, msg):
        nonlocal n_ok
        assert cond, msg
        n_ok += 1
    c12 = sieve_channels(12, "original")
    check(c12.tolist() == [2, 1, 1, 0, 2, 2, 1, 1, 0, 2, 2, 1], "original routing must match the Colab run")
    check((c12 == 0).astype(int).tolist() == [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0], "keep mask must match the Colab run")
    k = sieve_channels(5000, "original") == 0
    check(abs(k.mean() - 0.2) < 1e-9, "original keeps 1/5 over the first 5000 tokens, not 1/3")
    check(np.array_equal(k, stride_keep(5000, 5, 3)), "original is exactly stride-5 early on")
    check(abs(GOLDEN_ANGLE / 3 - 0.8) < 2e-5, "golden angle / 3 is almost 4/5")
    big = sieve_channels(1_000_000, "original")
    check(all(abs((big == j).mean() - 1 / 3) < 0.01 for j in range(3)), "original is 1/3 each only in the very long run")
    f = sieve_channels(100_000, "fixed")
    check(all(abs((f == j).mean() - 1 / 3) < 0.005 for j in range(3)), "fixed gives 1/3 per channel")
    check(abs((f[:50] == 0).mean() - 1 / 3) < 0.1, "fixed is about 1/3 even in short windows")
    check(alias_score(k) > 0.7, "original aliases badly")
    check(alias_score(f == 0) < 0.05, "fixed barely aliases")
    check(max_gap(f == 0) <= 5, "fixed has bounded gaps (three-gap property)")
    x = np.random.default_rng(1).random((2, 30, 8))
    s, ids, ch = sieve_select(x, "fixed")
    check(s.shape == (2, len(ids), 8) and np.array_equal(s, x[:, ids, :]), "select keeps the channel-0 tokens")
    check(len(ch) == 30 and len(ids) == (ch == 0).sum(), "position ids match the mask")
    check(np.array_equal(sieve_channels(30, "fixed"), sieve_channels(300, "fixed")[:30]), "mask for position n ignores total length (causal / streamable)")
    sc = np.random.default_rng(2).random(300)
    h = hybrid_keep(sc, 0.5)
    check(h.sum() == 150, "hybrid uses exactly the total budget")
    top = np.argsort(-sc)[:30]
    check(h[top].all() or h[top].mean() > 0.8, "hybrid keeps nearly all top-scored tokens")
    check(max_gap(hybrid_keep(np.zeros(300), 0.5)) <= 5, "with no scores the hybrid is still evenly spread (no big holes)")
    print(f"all {n_ok} checks passed")


if __name__ == "__main__":
    a = sys.argv
    if "--test" in a:
        run_tests()
    elif "--audit" in a:
        audit()
    elif "--compare" in a:
        compare()
    else:
        print(__doc__)


In [ ]:
%%writefile nkp_ledger.py
"""
nkp_ledger.py - the NKP claims ledger.  One file, Python standard library (sqlite3) + your nkp_math.py.

Every claim you check is stored once, with its verdict, its evidence, who made it, and what it depends on.

    python nkp_ledger.py --test                          self-checks; prints "all N checks passed" (empty = failure)
    python nkp_ledger.py seed-real                       check and store the 17 real claims from nkp_math_bench.py
    python nkp_ledger.py add identity "L" "R" [--source S] [--depends 1,2] [--domain '{"x":[0.1,3]}']
    python nkp_ledger.py add value "pi" "3.14159"        |  add dimensions "F = m*a" '{"F":"M L T^-2","m":"M","a":"L T^-2"}'
    python nkp_ledger.py add exceeds EXPR VAR LO HI C    |  add coincidence A B "formula"
    python nkp_ledger.py link CHILD PARENT               "CHILD is built on PARENT"
    python nkp_ledger.py list [STATUS]                   all claims, or only one verdict status
    python nkp_ledger.py search TEXT                     find claims by text (is this already known?)
    python nkp_ledger.py risk                            claims built on refuted (AT_RISK) or unverified (SHAKY) claims
    python nkp_ledger.py sources                         source track record
    python nkp_ledger.py recheck                         re-run every claim with the current engine, report any change
    python nkp_ledger.py stats | export claims.csv
    add  --db FILE   to any command to use a different database (default nkp_claims.db)

What "new" means here:  a claim is NEW only if its canonical form is not already stored.  Same claim written
differently (sides swapped, terms reordered) is a DUPLICATE.  A different claim about the same numeric function
is flagged RELATED.  Variable renames are not detected.  A stored claim is a verified statement of math, not a
fact about the world: the ledger records what computation found, with the evidence.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import os
import sqlite3
import sys
import time
from collections import defaultdict

import sympy as sp

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from nkp_math import ACCEPT, REJECT, NKPMath, parse  # noqa: E402

ARGS = {                                   # positional order of each claim kind (matches the engine)
    "identity": ["lhs", "rhs", "domain"],
    "value": ["expr", "claimed"],
    "exceeds": ["expr", "var", "lo", "hi", "c"],
    "dimensions": ["equation", "dims"],
    "coincidence": ["a", "b", "formula"],
}


def _engine_version() -> str:
    try:
        with open(os.path.join(HERE, "nkp_math.py"), "rb") as f:
            return hashlib.sha256(f.read()).hexdigest()[:10]
    except OSError:
        return "unknown"


def status_class(s: str) -> str:
    return "true" if s in ACCEPT else "false" if s in REJECT else "open"


# ------------------------------------------------------------------ canonical forms
def _fp(text: str):
    """Numeric fingerprint of an expression at fixed rational points (None if it cannot be evaluated)."""
    try:
        e = parse(text)
        syms = sorted(e.free_symbols, key=str)
        pts = {s: sp.Rational(7 + 6 * i, 10) for i, s in enumerate(syms)}
        return str(sp.N(e.subs(pts), 30))[:24] + "@" + ",".join(map(str, syms))
    except Exception:
        return None


def canonical(kind: str, kw: dict):
    """(canon, fingerprint, display).  canon decides DUPLICATE; fingerprint decides RELATED."""
    try:
        if kind == "identity":
            a, b = sorted([str(parse(kw["lhs"])), str(parse(kw["rhs"]))])
            dom = json.dumps(kw.get("domain"), sort_keys=True)
            fa, fb = _fp(kw["lhs"]), _fp(kw["rhs"])
            fp = None if None in (fa, fb) else "identity|" + "|".join(sorted([fa, fb])) + "|" + dom
            return f"identity|{a}|{b}|{dom}", fp, f"{kw['lhs']}  =  {kw['rhs']}" + (f"   [domain {dom}]" if kw.get("domain") else "")
        if kind == "value":
            c = f"value|{parse(kw['expr'])}|{kw['claimed']}"
            f = _fp(kw["expr"])
            return c, (None if f is None else f"value|{f}|{kw['claimed']}"), f"{kw['expr']}  =  {kw['claimed']}"
        if kind == "exceeds":
            c = f"exceeds|{parse(kw['expr'])}|{kw['var']}|{kw['lo']}|{kw['hi']}|{kw['c']}"
            f = _fp(kw["expr"])
            return c, (None if f is None else f"exceeds|{f}|{kw['var']}|{kw['lo']}|{kw['hi']}|{kw['c']}"), \
                f"|{kw['expr']}| > {kw['c']} on [{kw['lo']}, {kw['hi']}]"
        if kind == "dimensions":
            eq = kw["equation"]
            sides = sorted(str(parse(s)) for s in eq.split("=")) if "=" in eq else [eq]
            c = f"dimensions|{'=='.join(sides)}|{json.dumps(kw['dims'], sort_keys=True, default=str)}"
            return c, c, eq
        if kind == "coincidence":
            c = f"coincidence|{kw['a']}|{kw['b']}|{kw.get('formula', '')}"
            return c, c, f"{kw['a']} ~ {kw['b']}  via  {kw.get('formula', '') or '(no formula)'}"
    except Exception:
        pass
    raw = json.dumps(kw, sort_keys=True, default=str)
    return f"{kind}|RAW|{raw}", None, f"{kind}: {raw[:80]}"


def _revive(kind: str, kw: dict) -> dict:
    kw = dict(kw)
    if kind == "identity" and kw.get("domain"):
        kw["domain"] = {k: tuple(v) for k, v in kw["domain"].items()}
    return kw


# ------------------------------------------------------------------ the ledger
class ClaimLedger:
    def __init__(self, path: str = "nkp_claims.db", engine: NKPMath | None = None):
        self.path = path
        self.eng = engine or NKPMath(seed=0)
        self.ver = _engine_version()
        self.db = sqlite3.connect(path)
        self.db.executescript("""
            CREATE TABLE IF NOT EXISTS claims(
                id INTEGER PRIMARY KEY, kind TEXT NOT NULL, args TEXT NOT NULL, display TEXT NOT NULL,
                canon TEXT NOT NULL UNIQUE, fp TEXT, created TEXT NOT NULL);
            CREATE TABLE IF NOT EXISTS verdicts(
                id INTEGER PRIMARY KEY, claim_id INTEGER NOT NULL REFERENCES claims(id), status TEXT NOT NULL,
                detail TEXT, source TEXT, engine TEXT, ts TEXT NOT NULL);
            CREATE TABLE IF NOT EXISTS deps(
                child INTEGER NOT NULL, parent INTEGER NOT NULL, UNIQUE(child, parent));
            CREATE INDEX IF NOT EXISTS ix_fp ON claims(fp);
            CREATE INDEX IF NOT EXISTS ix_v ON verdicts(claim_id);
        """)
        self.db.commit()

    # ---- internals
    @staticmethod
    def _now() -> str:
        return time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())

    def _latest(self, cid: int):
        return self.db.execute("SELECT status, detail FROM verdicts WHERE claim_id=? ORDER BY id DESC LIMIT 1", (cid,)).fetchone()

    def _add_verdict(self, cid, status, detail, source):
        self.db.execute("INSERT INTO verdicts(claim_id,status,detail,source,engine,ts) VALUES(?,?,?,?,?,?)",
                        (cid, status, detail, source, self.ver, self._now()))

    def _link_many(self, child: int, parents):
        for p in parents:
            if p != child and self.db.execute("SELECT 1 FROM claims WHERE id=?", (p,)).fetchone():
                self.db.execute("INSERT OR IGNORE INTO deps(child,parent) VALUES(?,?)", (child, p))

    def _result(self, cid, novel, duplicate, related):
        s, d = self._latest(cid)
        r = self.risk().get(cid)
        return dict(id=cid, novel=novel, duplicate_of=cid if duplicate else None, related=related,
                    status=s, accepted=s in ACCEPT, detail=d, risk=r[0] if r else "")

    # ---- main entry
    def check(self, kind: str, source: str | None = None, depends_on=(), **kw) -> dict:
        if kind not in ARGS:
            raise ValueError(f"unknown claim kind {kind!r}; use one of {sorted(ARGS)}")
        canon, fp, display = canonical(kind, kw)
        row = self.db.execute("SELECT id FROM claims WHERE canon=?", (canon,)).fetchone()
        if row:                                                     # already known: no re-computation
            cid = row[0]
            if source is not None and not self.db.execute(
                    "SELECT 1 FROM verdicts WHERE claim_id=? AND source=?", (cid, source)).fetchone():
                s, d = self._latest(cid)
                self._add_verdict(cid, s, d, source)               # a new source asserted a known claim
            self._link_many(cid, depends_on)
            self.db.commit()
            return self._result(cid, novel=False, duplicate=True, related=[])
        v = getattr(self.eng, kind)(**kw, source=source)
        cur = self.db.execute("INSERT INTO claims(kind,args,display,canon,fp,created) VALUES(?,?,?,?,?,?)",
                              (kind, json.dumps(kw, default=str), display, canon, fp, self._now()))
        cid = cur.lastrowid
        self._add_verdict(cid, v.status, v.detail, source)
        self._link_many(cid, depends_on)
        related = [r[0] for r in self.db.execute("SELECT id FROM claims WHERE fp=? AND id!=?", (fp, cid))] if fp else []
        self.db.commit()
        return self._result(cid, novel=True, duplicate=False, related=related)

    def check_args(self, kind: str, args, source=None, depends_on=()):
        """Same, with positional arguments in the engine's order."""
        return self.check(kind, source=source, depends_on=depends_on, **dict(zip(ARGS[kind], args)))

    def link(self, child: int, parent: int) -> None:
        self._link_many(child, [parent]); self.db.commit()

    # ---- re-running
    def recheck(self, cid: int):
        kind, args = self.db.execute("SELECT kind,args FROM claims WHERE id=?", (cid,)).fetchone()
        old = self._latest(cid)[0]
        v = getattr(self.eng, kind)(**_revive(kind, json.loads(args)))
        self._add_verdict(cid, v.status, v.detail, None)           # source None: not counted in any source record
        self.db.commit()
        return v.status != old, old, v.status

    def recheck_all(self):
        out = []
        for (cid,) in self.db.execute("SELECT id FROM claims ORDER BY id").fetchall():
            changed, old, new = self.recheck(cid)
            if changed:
                out.append((cid, old, new))
        return out

    # ---- dependency risk
    def risk(self) -> dict:
        latest = {cid: self._latest(cid)[0] for (cid,) in self.db.execute("SELECT id FROM claims")}
        parents = defaultdict(list)
        for c, p in self.db.execute("SELECT child,parent FROM deps"):
            parents[c].append(p)
        out = {}
        for c in latest:
            seen, stack, bad, opn = set(), list(parents[c]), set(), set()
            while stack:
                p = stack.pop()
                if p in seen or p == c:
                    continue
                seen.add(p)
                cls = status_class(latest[p])
                if cls == "false":
                    bad.add(p)
                elif cls == "open":
                    opn.add(p)
                stack.extend(parents[p])
            if bad:
                out[c] = ("AT_RISK", sorted(bad), sorted(opn))
            elif opn:
                out[c] = ("SHAKY", [], sorted(opn))
        return out

    # ---- views
    def sources(self):
        k = defaultdict(lambda: {"true": 0, "false": 0, "open": 0})
        for s, st in self.db.execute("SELECT source,status FROM verdicts WHERE source IS NOT NULL"):
            k[s][status_class(st)] += 1
        rows = [(s, c["true"], c["false"], c["open"], (c["true"] + 1) / (c["true"] + c["false"] + 2)) for s, c in k.items()]
        return sorted(rows, key=lambda r: (-r[4], r[0]))

    def rows(self, status: str | None = None):
        out = []
        for cid, kind, display, created in self.db.execute("SELECT id,kind,display,created FROM claims ORDER BY id").fetchall():
            s, d = self._latest(cid)
            if status is None or s == status.upper():
                out.append((cid, kind, display, s, d, created))
        return out

    def search(self, text: str):
        like = f"%{text}%"
        ids = {r[0] for r in self.db.execute("SELECT id FROM claims WHERE display LIKE ? OR canon LIKE ?", (like, like))}
        return [r for r in self.rows() if r[0] in ids]

    def stats(self) -> dict:
        n = self.db.execute("SELECT COUNT(*) FROM claims").fetchone()[0]
        by = defaultdict(int)
        for r in self.rows():
            by[r[3]] += 1
        return dict(claims=n, by_status=dict(by), deps=self.db.execute("SELECT COUNT(*) FROM deps").fetchone()[0],
                    at_risk=sum(1 for v in self.risk().values() if v[0] == "AT_RISK"),
                    shaky=sum(1 for v in self.risk().values() if v[0] == "SHAKY"))

    def export_csv(self, path: str) -> int:
        risk = self.risk()
        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["id", "kind", "claim", "status", "detail", "sources", "risk", "created"])
            for cid, kind, display, s, d, created in self.rows():
                srcs = ";".join(sorted({r[0] for r in self.db.execute(
                    "SELECT source FROM verdicts WHERE claim_id=? AND source IS NOT NULL", (cid,))}))
                w.writerow([cid, kind, display, s, d, srcs, risk.get(cid, ("",))[0], created])
        return len(self.rows())


# ------------------------------------------------------------------ seeding with the real claims
def seed_real(led: ClaimLedger) -> list:
    from nkp_math_bench import REAL
    return [(src, led.check_args(kind, args, source=src)) for kind, args, _exp, src in REAL]


# ------------------------------------------------------------------ tests
def run_tests() -> None:
    n_ok = 0

    def check(cond, msg):
        nonlocal n_ok
        assert cond, msg
        n_ok += 1
    L = ClaimLedger(":memory:")
    a = L.check("identity", lhs="sin(x)**2+cos(x)**2", rhs="1", source="doc:A")
    check(a["novel"] and a["status"] == "PROVED" and a["accepted"], "true identity is stored as PROVED and novel")
    b = L.check("identity", lhs="1", rhs="cos(x)**2+sin(x)**2", source="doc:A")
    check(b["duplicate_of"] == a["id"] and not b["novel"], "sides swapped and terms reordered = DUPLICATE")
    check(L.stats()["claims"] == 1, "duplicate adds no claim")
    c = L.check("identity", lhs="cos(2*x)", rhs="1-2*sin(x)**2", source="doc:B")
    d = L.check("identity", lhs="cos(2*x)", rhs="2*cos(x)**2-1", source="doc:B")
    check(c["novel"] and d["novel"] and c["id"] in d["related"], "different claim about the same function is RELATED, not a duplicate")
    f = L.check("identity", lhs="(x+1)**2", rhs="x**2+3*x+1", source="doc:C")
    check(f["status"] == "REFUTED" and not f["accepted"], "false identity is REFUTED")
    g = L.check("identity", lhs="sqrt(x**2)", rhs="x", source="doc:C")
    check(g["status"] == "CONDITIONAL", "domain trap is CONDITIONAL")
    h = L.check("identity", lhs="x**2-4", rhs="(x-2)*(x+2)", source="doc:C", depends_on=[f["id"]])
    check(h["risk"] == "AT_RISK", "claim built on a refuted claim is AT_RISK")
    i = L.check("identity", lhs="x**3-8", rhs="(x-2)*(x**2+2*x+4)", source="doc:C", depends_on=[h["id"]])
    check(i["risk"] == "AT_RISK", "risk travels through a chain")
    j = L.check("identity", lhs="x*(1/x)", rhs="1", source="doc:D", depends_on=[g["id"]])
    check(j["risk"] == "SHAKY", "claim built on an unverified (conditional) claim is SHAKY")
    L.link(a["id"], j["id"]); L.link(j["id"], a["id"])
    check(isinstance(L.risk(), dict), "dependency cycles do not hang")
    ch = L.check("identity", lhs="1", rhs="cos(x)**2+sin(x)**2", source="doc:E")
    check(ch["duplicate_of"] == a["id"] and L.stats()["claims"] == 8, "second source on a known claim adds a verdict, not a claim")
    n0 = L.db.execute("SELECT COUNT(*) FROM verdicts").fetchone()[0]
    L.check("identity", lhs="1", rhs="cos(x)**2+sin(x)**2", source="doc:E")
    check(L.db.execute("SELECT COUNT(*) FROM verdicts").fetchone()[0] == n0, "same source repeating a claim is not double counted")
    t = {r[0]: r for r in L.sources()}
    check(t["doc:A"][1:4] == (1, 0, 0) and abs(t["doc:A"][4] - 2 / 3) < 1e-9, "source record: 1 true, reliability 2/3")
    check(t["doc:C"][1:4] == (2, 1, 1), "source record counts true / false / open separately")
    ok = L.check("value", expr="pi", claimed="3.14159265358979323846", source="doc:V")
    check(ok["status"] in ACCEPT, "value claim to 20 decimals")
    bad = L.check("value", expr="pi", claimed="3.14159265358979323847", source="doc:V")
    check(bad["status"] == "REFUTED", "value claim with last digit wrong is REFUTED")
    dm = L.check("dimensions", equation="F = m*a", dims={"F": "M L T^-1", "m": "M", "a": "L T^-2"}, source="doc:U")
    check(dm["status"] == "ILL_FORMED", "dimension claim with wrong units is ILL_FORMED")
    rc = L.recheck(a["id"])
    check(rc[0] is False, "recheck with the same engine changes nothing")
    L.db.execute("UPDATE verdicts SET status='REFUTED' WHERE claim_id=?", (a["id"],)); L.db.commit()
    rc2 = L.recheck(a["id"])
    check(rc2[0] is True and rc2[1] == "REFUTED" and rc2[2] == "PROVED", "recheck detects a changed verdict")
    check(len(L.search("sin")) >= 2 and len(L.search("zzzz")) == 0, "search finds stored claims by text")
    try:
        L.check("poem", x=1); check(False, "unknown kind must raise")
    except ValueError:
        check(True, "unknown kind raises ValueError")
    import tempfile
    p = os.path.join(tempfile.mkdtemp(), "t.db")
    P = ClaimLedger(p); P.check("identity", lhs="x*1", rhs="x", source="s"); P.db.close()
    P2 = ClaimLedger(p)
    check(P2.stats()["claims"] == 1 and P2.rows()[0][3] == "PROVED", "the database persists on disk")
    q = os.path.join(os.path.dirname(p), "e.csv")
    check(L.export_csv(q) == L.stats()["claims"] and os.path.getsize(q) > 100, "csv export")
    R = ClaimLedger(":memory:")
    from nkp_math_bench import REAL
    res = seed_real(R)
    exp = {"accept": ACCEPT, "reject": REJECT}
    ok_all = all((r["status"] in exp["accept"]) if e == "accept" else (r["status"] in exp["reject"]) if e == "reject"
                 else (r["status"] not in ACCEPT | REJECT) for (_k, _a, e, _s), (_s2, r) in zip(REAL, res))
    check(ok_all and len(res) == len(REAL), "all 17 real claims get the expected verdict through the ledger")
    check(R.stats()["claims"] == len({r["id"] for _s, r in res}), "every real claim is stored once")
    eng = NKPMath(seed=0)
    for kind, args, _e, src in REAL:
        getattr(eng, kind)(*args, source=src)
    mine = {r[0]: round(r[4], 9) for r in R.sources()}
    theirs = {r[0]: round(r[4], 9) for r in eng.ledger.table()}
    check(mine == theirs, "source reliability matches the engine's own ledger")
    print(f"all {n_ok} checks passed")


# ------------------------------------------------------------------ command line
def _fmt(r) -> str:
    cid, kind, display, s, d, created = r
    return f"#{cid:<3} {s:<12} {kind:<11} {display[:70]}"


def _cli_kw(kind, pos, domain_json):
    names = ARGS[kind]
    if kind == "identity":
        kw = dict(zip(names, pos))
        if domain_json:
            kw["domain"] = {k: tuple(v) for k, v in json.loads(domain_json).items()}
        return kw
    vals = list(pos)
    if kind == "dimensions" and len(vals) > 1:
        vals[1] = json.loads(vals[1])
    if kind == "coincidence":
        vals[:2] = [float(v) for v in vals[:2]]
    if kind == "exceeds" and len(vals) >= 4:
        vals[2], vals[3] = float(vals[2]), float(vals[3])
    return dict(zip(names, vals))


def main(argv) -> int:
    if "--test" in argv:
        run_tests()
        return 0
    ap = argparse.ArgumentParser(description="NKP claims ledger", add_help=True)
    ap.add_argument("--db", default="nkp_claims.db")
    sub = ap.add_subparsers(dest="cmd")
    p = sub.add_parser("add"); p.add_argument("kind"); p.add_argument("pos", nargs="+")
    p.add_argument("--source"); p.add_argument("--depends", default=""); p.add_argument("--domain")
    p = sub.add_parser("link"); p.add_argument("child", type=int); p.add_argument("parent", type=int)
    p = sub.add_parser("list"); p.add_argument("status", nargs="?")
    p = sub.add_parser("search"); p.add_argument("text")
    p = sub.add_parser("export"); p.add_argument("path")
    for n in ("seed-real", "risk", "sources", "recheck", "stats"):
        sub.add_parser(n)
    a = ap.parse_args(argv)
    if not a.cmd:
        print(__doc__)
        return 0
    L = ClaimLedger(a.db)
    if a.cmd == "add":
        deps = [int(x) for x in a.depends.split(",") if x.strip()]
        r = L.check(a.kind, source=a.source, depends_on=deps, **_cli_kw(a.kind, a.pos, a.domain))
        tag = "NEW" if r["novel"] else f"DUPLICATE of #{r['duplicate_of']}"
        print(f"#{r['id']} {tag}  {r['status']}  {r['detail'][:100]}")
        if r["related"]:
            print(f"   related (same numeric function): {r['related']}")
        if r["risk"]:
            print(f"   {r['risk']}")
    elif a.cmd == "link":
        L.link(a.child, a.parent); print(f"#{a.child} now depends on #{a.parent}")
    elif a.cmd == "seed-real":
        for src, r in seed_real(L):
            print(f"#{r['id']:<3} {'NEW' if r['novel'] else 'dup':<4} {r['status']:<12} {src}")
    elif a.cmd == "list":
        for r in L.rows(a.status): print(_fmt(r))
    elif a.cmd == "search":
        rows = L.search(a.text)
        print(f"{len(rows)} match(es)" if rows else "no match: nothing like this is stored yet")
        for r in rows: print(_fmt(r))
    elif a.cmd == "risk":
        rk = L.risk()
        print(f"{len(rk)} claim(s) built on refuted or unverified claims" if rk else "no claim is built on a refuted or unverified claim")
        for c, (lvl, bad, opn) in sorted(rk.items()):
            print(f"#{c:<3} {lvl:<8} refuted parents {bad or '-'}  unverified parents {opn or '-'}")
    elif a.cmd == "sources":
        for s, t, f, o, rel in L.sources(): print(f"{s:<36}{t} / {f} / {o}  -> {rel:.2f}")
    elif a.cmd == "recheck":
        ch = L.recheck_all()
        print("no verdict changed" if not ch else "CHANGED:")
        for c, old, new in ch: print(f"#{c}: {old} -> {new}")
    elif a.cmd == "stats":
        print(json.dumps(L.stats(), indent=1))
    elif a.cmd == "export":
        print(f"{L.export_csv(a.path)} claims written to {a.path}")
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv[1:]))


In [ ]:
%%writefile nkp_realtest.py
"""
nkp_realtest.py - pre-registered real-claims test for NKP-Math.  One file; needs nkp_math.py.

The point of this test is that the labels are fixed BEFORE the engine sees the claims.  The order is enforced:

  1. write claims.txt            one claim per line (format below), each with a label and where it came from
  2. python nkp_realtest.py check  claims.txt     syntax check only; runs nothing
  3. python nkp_realtest.py freeze claims.txt     locks claims + criteria with a fingerprint (sha256)
  4. python nkp_realtest.py run    claims.txt     refuses unless frozen and unchanged; scores once
  5. python nkp_realtest.py audit  claims.txt     lists every disagreement for review (no engine changes yet)
     python nkp_realtest.py --test                self-checks; prints "all N checks passed" (empty = failure)
     add  --ledger nkp_claims.db  to `run` to also store the claims in the claims ledger

CLAIM FILE FORMAT (plain text, pipes between fields; blank lines and lines starting with # are ignored)

  id | label | kind | source | arg1 | arg2 | ...   ## original wording of the claim

  label  true | false | conditional      what an INDEPENDENT calculation or reference says (not this engine)
  kind and args:
    identity    | lhs | rhs [| {"x":[0.1,3]}]            optional domain as JSON
    value       | expr | claimed-decimal-string
    exceeds     | expr | var | lo | hi | c                the claim is  max |expr| > c  on [lo, hi]
    dimensions  | equation | {"F":"M L T^-2","m":"M"}    dims as JSON
    coincidence | a | b | formula                          label false = "the match is NOT meaningful"
  Ids starting with EX are examples and are refused at freeze time.

PRE-REGISTERED CRITERIA are the text in CRITERIA below; they are part of the fingerprint, so changing them
after freezing makes `run` refuse.
"""
from __future__ import annotations

import csv
import hashlib
import json
import os
import sys
import tempfile
import time

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from nkp_math import ACCEPT, REJECT, NKPMath  # noqa: E402

CRITERIA = """PRE-REGISTERED CRITERIA (fixed before any claim is run)
SIZE   at least 50 claims, of which at least 15 are labelled false or conditional; otherwise the result is INCONCLUSIVE
R1     safety: ZERO false accepts (engine accepts a claim labelled false or conditional)
R2     agreement: of claims given a decided verdict (accept / reject / conditional), at least 90% match the label
R3     at most 5% of claims labelled true are rejected (false rejects)
R4     honesty: NOT_CHECKABLE (unparseable or unsupported) and UNDETERMINED are never counted as accepts;
       they are reported as coverage and not scored
SET    this set is run once.  Any later run of the same set is reported as RERUN and is no longer a pre-registered test.
       Fixes made after seeing results are judged on a NEW set, never on this one.
"""
MIN_N, MIN_NEG = 50, 15
LABELS = {"true", "false", "conditional"}
NARGS = {"identity": (2, 3), "value": (2, 2), "exceeds": (5, 5), "dimensions": (2, 2), "coincidence": (3, 3)}
ARGNAMES = {"identity": ["lhs", "rhs", "domain"], "value": ["expr", "claimed"],
            "exceeds": ["expr", "var", "lo", "hi", "c"], "dimensions": ["equation", "dims"],
            "coincidence": ["a", "b", "formula"]}


# ------------------------------------------------------------------ parsing
def parse_claims(text: str):
    rows = []
    for no, raw in enumerate(text.splitlines(), 1):
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        body, _, original = line.partition(" ## ")
        f = [p.strip() for p in body.split("|")]
        if len(f) < 5:
            raise ValueError(f"line {no}: need  id | label | kind | source | args...")
        cid, label, kind, source, args = f[0], f[1].lower(), f[2].lower(), f[3], f[4:]
        if label not in LABELS:
            raise ValueError(f"line {no}: label must be true, false or conditional (got {label!r})")
        if kind not in NARGS:
            raise ValueError(f"line {no}: kind must be one of {sorted(NARGS)} (got {kind!r})")
        lo, hi = NARGS[kind]
        if not lo <= len(args) <= hi:
            raise ValueError(f"line {no}: {kind} takes {lo}..{hi} arguments, got {len(args)}")
        kw = dict(zip(ARGNAMES[kind], args))
        try:
            if kind == "identity" and "domain" in kw:
                kw["domain"] = {k: tuple(v) for k, v in json.loads(kw["domain"]).items()}
            if kind == "dimensions":
                kw["dims"] = json.loads(kw["dims"])
            if kind == "coincidence":
                kw["a"], kw["b"] = float(kw["a"]), float(kw["b"])
            if kind == "exceeds":
                kw["lo"], kw["hi"] = float(kw["lo"]), float(kw["hi"])
        except Exception as e:
            raise ValueError(f"line {no}: could not read an argument ({type(e).__name__}: {e})")
        rows.append(dict(id=cid, label=label, kind=kind, source=source, kw=kw, original=original.strip(), line=no))
    ids = [r["id"] for r in rows]
    if len(set(ids)) != len(ids):
        raise ValueError("duplicate claim ids: " + ", ".join(sorted({i for i in ids if ids.count(i) > 1})))
    return rows


def fingerprint(text: str, criteria: str = CRITERIA) -> str:
    return hashlib.sha256((criteria + "\n=====\n" + text).encode("utf-8")).hexdigest()


def _frozen_path(p): return p + ".frozen"
def _log_path(p): return os.path.join(os.path.dirname(os.path.abspath(p)), "realtest_runs.log")


# ------------------------------------------------------------------ scoring
def engine_verdict(eng: NKPMath, row):
    """-> (class, status, detail).  class: accept | reject | conditional | undetermined | not_checkable.  Never guesses."""
    try:
        v = getattr(eng, row["kind"])(**row["kw"], source=row["source"])
    except Exception as e:
        return "not_checkable", "NOT_CHECKABLE", f"{type(e).__name__}: {e}"[:120]
    if v.status in ACCEPT:
        return "accept", v.status, v.detail
    if v.status in REJECT:
        return "reject", v.status, v.detail
    if v.status == "CONDITIONAL":
        return "conditional", v.status, v.detail
    return "undetermined", v.status, v.detail


WANT = {"true": "accept", "false": "reject", "conditional": "conditional"}


def evaluate(rows, results, criteria_ok=True) -> dict:
    n = len(rows)
    neg = sum(r["label"] != "true" for r in rows)
    fa = [r["id"] for r, x in zip(rows, results) if r["label"] != "true" and x[0] == "accept"]
    t_rows = [(r, x) for r, x in zip(rows, results) if r["label"] == "true"]
    fr = [r["id"] for r, x in t_rows if x[0] == "reject"]
    decided = [(r, x) for r, x in zip(rows, results) if x[0] in ("accept", "reject", "conditional")]
    agree = [r["id"] for r, x in decided if x[0] == WANT[r["label"]]]
    disagree = [r["id"] for r, x in decided if x[0] != WANT[r["label"]]]
    out = dict(n=n, neg=neg, decided=len(decided), coverage=len(decided) / n if n else 0.0,
               not_checkable=sum(x[0] == "not_checkable" for x in results),
               undetermined=sum(x[0] == "undetermined" for x in results),
               false_accepts=fa, false_rejects=fr, disagree=disagree,
               agreement=(len(agree) / len(decided)) if decided else 0.0,
               false_reject_rate=(len(fr) / len(t_rows)) if t_rows else 0.0)
    out["size_ok"] = n >= MIN_N and neg >= MIN_NEG
    out["R1"] = not fa
    out["R2"] = out["agreement"] >= 0.90 and len(decided) > 0
    out["R3"] = out["false_reject_rate"] <= 0.05
    out["verdict"] = ("INCONCLUSIVE (too few claims or too few false/conditional claims)" if not out["size_ok"]
                      else "PASS" if out["R1"] and out["R2"] and out["R3"] else "FAIL")
    return out


# ------------------------------------------------------------------ commands
def read(path) -> str:
    with open(path, encoding="utf-8") as f:
        return f.read()


def cmd_check(path):
    rows = parse_claims(read(path))
    ex = [r["id"] for r in rows if r["id"].upper().startswith("EX")]
    from collections import Counter
    c = Counter((r["kind"], r["label"]) for r in rows)
    print(f"{len(rows)} claims read.  by kind/label: " + ", ".join(f"{k[0]}:{k[1]}={v}" for k, v in sorted(c.items())))
    neg = sum(r["label"] != "true" for r in rows)
    print(f"false/conditional claims: {neg}   (need >= {MIN_NEG}; total need >= {MIN_N})")
    if ex:
        print(f"NOTE: example ids still present {ex}: delete them before freezing")
    return 0


def cmd_freeze(path):
    text = read(path)
    rows = parse_claims(text)
    ex = [r["id"] for r in rows if r["id"].upper().startswith("EX")]
    if ex:
        print(f"REFUSED: example claims are still in the file {ex}.  Delete them, then freeze.")
        return 1
    fp = fingerprint(text)
    with open(_frozen_path(path), "w") as f:
        json.dump(dict(sha256=fp, n=len(rows), frozen=time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())), f)
    print(f"FROZEN: {len(rows)} claims.  fingerprint {fp[:16]}...  Labels and criteria can no longer change without detection.")
    print(CRITERIA)
    return 0


def cmd_run(path, ledger_path=None):
    if not os.path.exists(_frozen_path(path)):
        print("REFUSED: the claim file is not frozen.  Run `freeze` first (that is what makes this a pre-registered test).")
        return 1
    text = read(path)
    fz = json.load(open(_frozen_path(path)))
    fp = fingerprint(text)
    if fp != fz["sha256"]:
        print("REFUSED: the claim file or the criteria changed after freezing.  Results would not be pre-registered.")
        return 1
    rows = parse_claims(text)
    log = _log_path(path)
    seen = os.path.exists(log) and any(json.loads(l).get("sha256") == fp for l in open(log) if l.strip())
    eng = NKPMath(seed=0)
    results = [engine_verdict(eng, r) for r in rows]
    ev = evaluate(rows, results)
    print(f"{'id':<8}{'label':<12}{'engine':<14}{'ok':<5}claim")
    for r, x in zip(rows, results):
        ok = "ok" if x[0] == WANT[r["label"]] else "--" if x[0] in ("undetermined", "not_checkable") else "BAD"
        name = r["original"] or " | ".join(str(v) for v in r["kw"].values())
        print(f"{r['id']:<8}{r['label']:<12}{x[0]:<14}{ok:<5}{name[:60]}")
    print(f"\n{ev['n']} claims; {ev['decided']} decided ({ev['coverage']:.0%} coverage); "
          f"{ev['undetermined']} undetermined, {ev['not_checkable']} not checkable (reported, not scored)")
    print(f"R1 false accepts: {len(ev['false_accepts'])} {ev['false_accepts'] or ''} -> {'PASS' if ev['R1'] else 'FAIL'}")
    print(f"R2 agreement on decided claims: {ev['agreement']:.1%} (need >= 90%) -> {'PASS' if ev['R2'] else 'FAIL'}")
    print(f"R3 false-reject rate on true claims: {ev['false_reject_rate']:.1%} (need <= 5%) -> {'PASS' if ev['R3'] else 'FAIL'}")
    print(f"\nRESULT: {ev['verdict']}" + ("   [RERUN: this set was run before, so this is NOT a pre-registered result]" if seen else ""))
    if ev["disagree"] or ev["false_accepts"]:
        print("Disagreements are NOT automatically engine errors: the label may be wrong.  Run `audit`, check labels first.")
    with open(log, "a") as f:
        f.write(json.dumps(dict(sha256=fp, time=time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), verdict=ev["verdict"],
                                n=ev["n"], false_accepts=ev["false_accepts"], agreement=round(ev["agreement"], 4))) + "\n")
    out_csv = path + ".results.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["id", "label", "kind", "source", "engine_class", "status", "detail", "original"])
        for r, x in zip(rows, results):
            w.writerow([r["id"], r["label"], r["kind"], r["source"], x[0], x[1], x[2], r["original"]])
    print(f"results written to {out_csv}")
    if ledger_path:
        from nkp_ledger import ClaimLedger
        L = ClaimLedger(ledger_path)
        stored = 0
        for r, x in zip(rows, results):
            if x[0] != "not_checkable":
                L.check(r["kind"], source=r["source"], **r["kw"]); stored += 1
        print(f"{stored} claims stored in the ledger {ledger_path}")
    return 0


def cmd_audit(path):
    rows = parse_claims(read(path))
    eng = NKPMath(seed=0)
    n = 0
    for r in rows:
        c, s, d = engine_verdict(eng, r)
        if c != WANT[r["label"]]:
            n += 1
            print(f"{r['id']}: label {r['label']}, engine {c} ({s})\n    claim:  {r['original'] or r['kw']}\n    engine: {d[:140]}\n"
                  f"    source: {r['source']}")
    print(f"\n{n} claim(s) to review.  Check the LABEL with an independent calculation first.  "
          f"Do not change the engine on this set; use a new set.")
    return 0


# ------------------------------------------------------------------ tests
def run_tests() -> None:
    n_ok = 0

    def check(cond, msg):
        nonlocal n_ok
        assert cond, msg
        n_ok += 1
    good = ("# comment\n\n"
            "c1 | true | identity | src:a | sin(x)**2+cos(x)**2 | 1 ## sin^2+cos^2=1\n"
            "c2 | false | identity | src:a | (x+1)**2 | x**2+1\n"
            "c3 | conditional | identity | src:b | sqrt(x**2) | x\n"
            "c4 | true | value | src:c | pi | 3.14159\n"
            "c5 | false | value | src:c | pi | 3.14160\n"
            "c6 | true | dimensions | src:d | F = m*a | {\"F\":\"M L T^-2\",\"m\":\"M\",\"a\":\"L T^-2\"}\n"
            "c7 | false | exceeds | src:e | sin(x)*exp(-x) | x | 0 | 10 | 0.35\n"
            "c8 | false | coincidence | src:f | 19.99909997 | 20.0 | exp(pi)-pi\n"
            "c9 | true | identity | src:g | sqrt(y**2) | y | {\"y\":[0.1,3]}\n")
    rows = parse_claims(good)
    check(len(rows) == 9 and rows[0]["original"] == "sin^2+cos^2=1", "claims and original wording parsed")
    check(rows[8]["kw"]["domain"] == {"y": (0.1, 3.0)}, "domain JSON becomes a tuple")
    for bad, why in [("c1 | maybe | value | s | pi | 3", "bad label"), ("c1 | true | poem | s | a", "bad kind"),
                     ("c1 | true | value | s | pi", "wrong argument count"), ("a | true | value | s | pi | 3\na | true | value | s | e | 2", "duplicate id")]:
        try:
            parse_claims(bad); check(False, f"{why} must be rejected")
        except ValueError:
            check(True, f"{why} is rejected")
    eng = NKPMath(seed=0)
    res = [engine_verdict(eng, r) for r in rows]
    got = [x[0] for x in res]
    check(got == ["accept", "reject", "conditional", "accept", "reject", "accept", "reject", "reject", "accept"], f"verdict classes {got}")
    ev = evaluate(rows, res)
    check(ev["R1"] and ev["R2"] and ev["R3"] and not ev["size_ok"] and ev["verdict"].startswith("INCONCLUSIVE"),
          "small set is INCONCLUSIVE even when everything agrees")
    wrong = [dict(r) for r in rows]; wrong[1]["label"] = "true"; wrong[1] = dict(wrong[1])
    flipped = [dict(r) for r in rows]; flipped[0]["label"] = "false"
    ev2 = evaluate(flipped, res)
    check(ev2["false_accepts"] == ["c1"] and not ev2["R1"], "an accepted claim labelled false is a false accept")
    ev3 = evaluate(wrong, res)
    check(ev3["false_rejects"] == ["c2"], "a rejected claim labelled true is a false reject")
    junk = dict(id="j1", label="true", kind="identity", source="s", kw=dict(lhs="foo(", rhs="1"), original="", line=1)
    jr = engine_verdict(eng, junk)
    check(jr[0] == "not_checkable", "unparseable claim is NOT_CHECKABLE, never guessed")
    ev4 = evaluate([junk], [jr])
    check(ev4["decided"] == 0 and not ev4["R2"], "not-checkable claims are not scored as accepts")
    check(fingerprint("a") != fingerprint("b") and fingerprint("a") != fingerprint("a", criteria="other"), "fingerprint covers claims and criteria")
    d = tempfile.mkdtemp(); p = os.path.join(d, "claims.txt")
    open(p, "w").write(good)
    check(cmd_run(p) == 1, "run refuses before freeze")
    ex = os.path.join(d, "ex.txt"); open(ex, "w").write("EX1 | true | value | s | pi | 3.14")
    check(cmd_freeze(ex) == 1, "freeze refuses example ids")
    check(cmd_freeze(p) == 0, "freeze works")
    import contextlib, io
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        r1 = cmd_run(p)
    check(r1 == 0 and "RESULT: INCONCLUSIVE" in buf.getvalue() and "RERUN" not in buf.getvalue(), "first run is not flagged")
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        cmd_run(p)
    check("RERUN" in buf.getvalue(), "second run of the same set is flagged RERUN")
    check(os.path.exists(p + ".results.csv"), "results csv written")
    open(p, "a").write("c10 | true | value | s | e | 2.71\n")
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        r3 = cmd_run(p)
    check(r3 == 1 and "REFUSED" in buf.getvalue(), "editing the file after freezing makes run refuse")
    print(f"all {n_ok} checks passed")


def main(argv) -> int:
    if "--test" in argv:
        run_tests(); return 0
    if len(argv) < 2 or argv[0] not in ("check", "freeze", "run", "audit"):
        print(__doc__); return 0
    cmd, path = argv[0], argv[1]
    try:
        if cmd == "check": return cmd_check(path)
        if cmd == "freeze": return cmd_freeze(path)
        if cmd == "audit": return cmd_audit(path)
        led = argv[argv.index("--ledger") + 1] if "--ledger" in argv else None
        return cmd_run(path, led)
    except (ValueError, FileNotFoundError) as e:
        print(f"ERROR: {e}"); return 1


if __name__ == "__main__":
    sys.exit(main(sys.argv[1:]))


In [ ]:
%%writefile real_claims_template.txt
# real_claims_template.txt  - copy to claims.txt, DELETE the EX lines, add your claims.
# Format:  id | label | kind | source | args...   ## original wording
# label = what an INDEPENDENT calculation or reference says: true | false | conditional
# source = where the claim came from, e.g.  gemini:2026-10-05  copilot:q3  textbook:stewart  user:own
# Include claims you suspect are wrong. Do not pick only claims you expect the engine to pass.
EX1 | true  | identity    | example | sin(2*x) | 2*sin(x)*cos(x) ## double-angle formula
EX2 | false | identity    | example | (x+y)**2 | x**2+y**2 ## a common mistake
EX3 | conditional | identity | example | log(x**2) | 2*log(x) ## only true for x>0
EX4 | true  | value       | example | pi | 3.14159265 ## pi to 8 decimals
EX5 | false | value       | example | exp(1) | 2.71829 ## e, last digit wrong
EX6 | true  | dimensions  | example | En = m*v**2/2 | {"En":"M L^2 T^-2","m":"M","v":"L T^-1"} ## kinetic energy
EX7 | false | dimensions  | example | En = m*v | {"En":"M L^2 T^-2","m":"M","v":"L T^-1"} ## wrong units
EX8 | true  | exceeds     | example | x*exp(-x) | x | 0 | 10 | 0.36 ## max of x e^-x is 0.3679
EX9 | false | coincidence | example | 6.0 | 6.02 | 2*3 ## numerology


## Part B - checks (cells 9-11)

In [ ]:
# 9. the referee's own tests (should print: all 28 checks passed)
!python nkp_math.py --test

In [ ]:
# 10. THE GATE: run after every AI edit (should end: RESULT: ALL PASS; includes ledger + real-test harness tests)
!python nkp_check.py

In [ ]:
# 11. real claims from your documents (should end: M4: 17 of 17 ... PASS)
!python nkp_math_bench.py --real

## Part C - dev benchmark (cells 12-13)

In [ ]:
# 12. dev run, seeds 0-9 (about 20 seconds)
!python nkp_math_bench.py --seeds 0:10 --out dev.json

In [ ]:
# 13. dev report (M1-M3 should say PASS)
!python nkp_math_bench.py --report dev.json

## Part D - sieve (cells 14-15)

In [ ]:
# 14. sieve self-test (should print: all 17 checks passed)
!python sieve_lab.py --test

In [ ]:
# 15. sieve audit and comparison
!python sieve_lab.py --audit
!python sieve_lab.py --compare

## Part E - claims ledger (cells 16-17)
The database `nkp_claims.db` is erased when Colab closes: export a CSV and download it.

In [ ]:
# 16. ledger self-test (should print: all 26 checks passed)
!python nkp_ledger.py --test

In [ ]:
# 17. fill the ledger with your 17 real claims, then export
!python nkp_ledger.py seed-real
!python nkp_ledger.py risk
!python nkp_ledger.py export claims.csv

## Part G - the real-claims test (cells 18-23)
**Only after claims.txt is filled in.** Order matters, and it is enforced:
1. cell 18 copies the template to `claims.txt`. Open `claims.txt` in the Files panel, delete the EX lines, add your claims with labels.
2. cell 19 checks the syntax (runs nothing).
3. cell 20 FREEZES the claims and criteria. After this, labels cannot change.
4. cell 21 runs the test ONCE.
5. cell 22 lists disagreements for review. **Do not edit the engine on this set.**
6. cell 23 saves everything (download `claims.txt.results.csv`, `realtest_runs.log`).

In [ ]:
# 18. start your claim file from the template (does nothing if claims.txt already exists)
!test -f claims.txt && echo "claims.txt already exists, left alone" || (cp real_claims_template.txt claims.txt && echo "claims.txt created from the template")

In [ ]:
# 19. syntax check only
!python nkp_realtest.py check claims.txt

In [ ]:
# 20. FREEZE (point of no return for labels)
!python nkp_realtest.py freeze claims.txt

In [ ]:
# 21. RUN ONCE
!python nkp_realtest.py run claims.txt --ledger nkp_claims.db

In [ ]:
# 22. review disagreements (check the LABELS first)
!python nkp_realtest.py audit claims.txt

In [ ]:
# 23. list what to download
!ls -la claims.txt claims.txt.frozen claims.txt.results.csv realtest_runs.log nkp_claims.db